In [7]:
import time
import pandas as pd
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from tqdm import tqdm

# ------------------------ COOKIE MANAGEMENT ------------------------ #
COOKIE_FILE = "maukerja_cookies.json"

def save_cookies(driver, filepath=COOKIE_FILE):
    """Save cookies to a file after login."""
    with open(filepath, 'w') as f:
        json.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {filepath}")

def load_cookies(driver, filepath=COOKIE_FILE):
    """Load cookies from file to maintain login session."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            cookies = json.load(f)
            for cookie in cookies:
                # Remove 'expiry' if it exists (can cause issues)
                if 'expiry' in cookie:
                    del cookie['expiry']
                try:
                    driver.add_cookie(cookie)
                except Exception as e:
                    print(f"⚠️ Could not add cookie: {e}")
        print(f"✅ Cookies loaded from {filepath}")
        return True
    else:
        print(f"⚠️ No cookie file found at {filepath}")
        return False

# ------------------------ SETUP ------------------------ #
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.page_load_strategy = "eager"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page={page}"

# ------------------------ MANUAL LOGIN ------------------------ #
print("\n" + "="*60)
print("🔐 MANUAL LOGIN REQUIRED")
print("="*60)

# Open Maukerja homepage
driver.get("https://www.maukerja.my/en")
print("⏳ Maukerja homepage loaded.")
time.sleep(3)

# Try to load existing cookies first
if load_cookies(driver):
    print("🔄 Refreshing page with saved cookies...")
    driver.refresh()
    time.sleep(5)
    print("✅ Page refreshed. Check if you're logged in.")
    print("   If not logged in, you'll need to sign in manually.")
else:
    print("ℹ️  No saved cookies found. You need to log in manually.")

print("\n📋 INSTRUCTIONS:")
print("   1. Check if you're already logged in (look for your profile icon)")
print("   2. If NOT logged in:")
print("      - Click the 'Sign In' or 'Log In' button")
print("      - Enter your email and password")
print("      - Complete any verification if required")
print("   3. Once logged in, return to this console")
print("\n⚠️  DO NOT close the browser window!")
print("="*60)

# Wait for user to manually log in
input("\n✋ Press ENTER after you have successfully logged in... ")

# Save cookies for future use
save_cookies(driver)

print("\n✅ Login confirmed! Starting scraping process...")
print("="*60 + "\n")

# Small delay to ensure session is stable
time.sleep(3)

# ------------------------ SAFE GET ------------------------ #
def safe_get(url, retries=3, wait_time=5):
    """Try loading a page with retries to avoid timeout crashes."""
    for i in range(retries):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"⚠️ Retry {i+1}/{retries} for {url} → {e}")
            time.sleep(wait_time)
    return False

# ------------------------ SCRAPE JOB DESCRIPTION ------------------------ #
def scrape_job_details(job_url):
    """Scrape detailed job information from the job detail page."""
    details = {
        "working_location": "Not Found",
        "requirements": "Not Found",
        "responsibilities": "Not Found",
        "benefits": "Not Found",
        "skills": "Not Found",
        "posted_date": "Not Found",
        "closing_date": "Not Found",
        "company_type": "Not Found",
        "company_size": "Not Found",
        "industry": "Not Found",
        "company_ssm_no": "Not Found",
        "job_vacancies": "Not Found",
        "company_website": "Not Found",
        "company_social_media": "Not Found"
    }
    
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])

        if not safe_get(job_url):
            return details

        time.sleep(3)  # Wait for modal/page to load
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # --- Extract Company Information ---
        try:
            # Find the company info container - try multiple selectors
            company_container = soup.find('div', class_='has-background-grey90 p-5 rounded-xl text-base border border-solid border-gray-100') or \
                               soup.find('div', class_='columns is-mobile is-multiline')
            
            if company_container:
                print(f"✅ Found company container")
                # Find all column divs that contain the information
                columns = company_container.find_all('div', class_='column is-6-mobile is-4-tablet')
                print(f"✅ Found {len(columns)} columns in company info")
                
                for col in columns:
                    # Get the header (h3) text to identify the field
                    header = col.find('h3', class_='has-text-weight-bold')
                    if header:
                        field_name = header.get_text(strip=True).lower()
                        print(f"   Processing field: {field_name}")
                        
                        # Get the value (p tag)
                        value_p = col.find('p')
                        
                        if 'company type' in field_name:
                            if value_p:
                                details["company_type"] = value_p.get_text(strip=True)
                                print(f"   ✓ Company Type: {details['company_type']}")
                        
                        elif 'company size' in field_name:
                            if value_p:
                                details["company_size"] = value_p.get_text(strip=True)
                                print(f"   ✓ Company Size: {details['company_size']}")
                        
                        elif 'industry' in field_name:
                            if value_p:
                                details["industry"] = value_p.get_text(strip=True)
                                print(f"   ✓ Industry: {details['industry']}")
                        
                        elif 'ssm' in field_name:
                            if value_p:
                                ssm = value_p.get_text(strip=True)
                                details["company_ssm_no"] = ssm if ssm != '-' else "Not Provided"
                                print(f"   ✓ SSM No: {details['company_ssm_no']}")
                        
                        elif 'job vacancies' in field_name or 'vacancies' in field_name:
                            if value_p:
                                details["job_vacancies"] = value_p.get_text(strip=True)
                                print(f"   ✓ Job Vacancies: {details['job_vacancies']}")
                        
                        elif 'company website' in field_name or ('website' in field_name and 'social' not in field_name):
                            # Look for link in p tag
                            if value_p:
                                link = value_p.find('a')
                                if link:
                                    details["company_website"] = link.get('href', 'Not Found')
                                    print(f"   ✓ Company Website: {details['company_website']}")
                        
                        elif 'social media' in field_name:
                            # Find all social media links in the column
                            social_div = col.find('div', class_='flex-wrap')
                            if social_div:
                                links = social_div.find_all('a', rel='nofollow')
                                if links:
                                    social_links = [link.get('href', '') for link in links if link.get('href')]
                                    details["company_social_media"] = "; ".join(social_links) if social_links else "Not Found"
                                    print(f"   ✓ Social Media: Found {len(social_links)} links")
                
                print(f"✅ Company information extraction complete")
            else:
                print(f"⚠️ Company container not found")
        except Exception as e:
            print(f"⚠️ Error extracting company information: {e}")
            import traceback
            traceback.print_exc()
        
        # --- Extract Working Location ---
        try:
            location_section = soup.find('div', {'id': 'job-locations'})
            if location_section:
                location_list = location_section.find('ul')
                if location_list:
                    locations = [li.get_text(strip=True) for li in location_list.find_all('li')]
                    details["working_location"] = "; ".join(locations)
        except Exception as e:
            print(f"⚠️ Error extracting working location: {e}")
        
        # --- Extract Job Description sections ---
        try:
            job_desc_section = soup.find('div', {'id': 'job-description'})
            if job_desc_section:
                # Find all sections within job description
                sections = job_desc_section.find_all('div', class_='space-y-4')
                
                for section in sections:
                    # Get section title
                    title_elem = section.find('p', class_='font-bold')
                    if title_elem:
                        title = title_elem.get_text(strip=True).lower()
                        
                        # Find the content div
                        content_div = section.find('div', class_='job-detail-text')
                        if content_div:
                            # Extract text from lists and paragraphs
                            content_parts = []
                            
                            # Get all ul/li elements
                            for ul in content_div.find_all('ul'):
                                items = [li.get_text(strip=True) for li in ul.find_all('li')]
                                content_parts.extend(items)
                            
                            # Get all paragraph elements
                            for p in content_div.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            
                            content = "\n".join(content_parts)
                            
                            # Categorize based on title
                            if 'requirement' in title:
                                details["requirements"] = content
                            elif 'responsibilit' in title:
                                details["responsibilities"] = content
                            elif 'benefit' in title:
                                details["benefits"] = content
        except Exception as e:
            print(f"⚠️ Error extracting job description sections: {e}")
        
        # --- Extract Skills ---
        try:
            print(f"🔍 Starting skills extraction...")
            skills_list = []
            
            # Method 1: Find via job-detail-text container
            job_detail_containers = soup.find_all('div', class_='class="job-detail-text text-base"')
            print(f"   Found {len(job_detail_containers)} job-detail-text containers")
            
            for container in job_detail_containers:
                # Check if this container has the flex wrapper for skills
                flex_div = container.find('div', class_='is-flex is-flex-wrap-wrap')
                if flex_div:
                    print(f"   Found flex wrapper in job-detail-text")
                    # Get all spans with the skill styling
                    skill_spans = flex_div.find_all('span', class_='px-2 py-1 has-text-weight-bold is-block border border-solid border-gray-100')
                    print(f"   Found {len(skill_spans)} skill spans")
                    
                    if skill_spans:
                        for span in skill_spans:
                            skill_text = span.get_text(strip=True)
                            if skill_text and len(skill_text) > 0:
                                skills_list.append(skill_text)
                        
                        if skills_list:
                            details["skills"] = ", ".join(skills_list)
                            print(f"✅ Found {len(skills_list)} skills: {', '.join(skills_list[:3])}...")
                            break
            
            # Method 2: Direct search for is-flex-wrap-wrap with spans
            if not skills_list:
                print(f"   Trying direct flex-wrap search...")
                all_flex_divs = soup.find_all('div', class_='is-flex is-flex-wrap-wrap')
                print(f"   Found {len(all_flex_divs)} flex-wrap divs")
                
                for flex_div in all_flex_divs:
                    # Check if this div has skill-like spans (with border styling)
                    skill_spans = flex_div.find_all('span', class_='px-2 py-1 has-text-weight-bold is-block border border-solid border-gray-100')
                    
                    # Filter to only get spans with border styling (skills, not other elements)
                    for span in skill_spans:
                        style = span.get('style', '')
                        if 'border-radius' in style or 'border: 1px solid' in style:
                            skill_text = span.get_text(strip=True)
                            if skill_text and len(skill_text) > 2:  # Avoid empty or very short text
                                skills_list.append(skill_text)
                    
                    if skills_list:
                        details["skills"] = ", ".join(skills_list)
                        print(f"✅ Found {len(skills_list)} skills (method 2): {', '.join(skills_list[:3])}...")
                        break
            
            # Method 3: Find by Skills section header
            if not skills_list:
                print(f"   Trying section header search...")
                all_sections = soup.find_all('div', class_='space-y-6')
                for section in all_sections:
                    header = section.find('p', class_='font-bold')
                    if header and 'skill' in header.get_text(strip=True).lower():
                        print(f"   Found Skills section header")
                        # Look for any spans with skills
                        all_spans = section.find_all('span', class_='px-2')
                        for span in all_spans:
                            skill_text = span.get_text(strip=True)
                            if skill_text and len(skill_text) > 2:
                                skills_list.append(skill_text)
                        
                        if skills_list:
                            details["skills"] = ", ".join(skills_list)
                            print(f"✅ Found {len(skills_list)} skills (method 3): {', '.join(skills_list[:3])}...")
                            break
            
            if not skills_list:
                print(f"⚠️ No skills found after trying all methods")
        except Exception as e:
            print(f"⚠️ Error extracting skills: {e}")
            import traceback
            traceback.print_exc()
        
        # --- Extract Posted and Closing Dates ---
        try:
            date_text = soup.find('div', class_='text-base', string=lambda text: text and 'Posted' in text)
            if date_text:
                date_str = date_text.get_text(strip=True)
                # Example: "Posted 2 minutes ago • Closing 7 Jan 2026"
                if 'Posted' in date_str:
                    parts = date_str.split('•')
                    if len(parts) >= 1:
                        details["posted_date"] = parts[0].replace('Posted', '').strip()
                    if len(parts) >= 2:
                        details["closing_date"] = parts[1].replace('Closing', '').strip()
        except Exception as e:
            print(f"⚠️ Error extracting dates: {e}")

    except Exception as e:
        print(f"❌ Error fetching job details for {job_url}: {e}")
    finally:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(1)
    
    return details

# ------------------------ MAIN SCRAPER ------------------------ #
max_pages = 2  # Testing with 2 pages first
all_jobs = []

for page in tqdm(range(1, max_pages + 1), desc="Pages", unit="page"):
    url = BASE_URL.format(page=page)
    print(f"\n📄 Loading Page {page} → {url}")

    if not safe_get(url):
        print(f"❌ Failed to load page {page}, skipping.")
        continue

    time.sleep(3)

    try:
        # CORRECTED: Target the main job card container
        job_card_selector = "div.job-card-lite"
        
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector)))
        job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
        print(f"🔍 Found {len(job_cards)} job cards on page {page}")

        for idx in tqdm(range(len(job_cards)), desc=f"Page {page}", unit="job", leave=False):
            try:
                # Re-find job_cards to avoid stale element reference
                job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
                job = job_cards[idx]
                
                # Initialize default values
                job_title = "Not Found"
                job_url = "Not Found"
                company_name = "Not Found"
                location = "Not Found"
                salary = "Not Found"
                job_type = "Not Found"
                listing_date = "Not Found"
                category_text = "Not Found"
                subcategory_text = "N/A"
                job_id = "Not Found"
                
                # --- Job Title and URL ---
                try:
                    title_link = job.find_element(By.CSS_SELECTOR, "div.font-bold a.has-text-dark")
                    job_title = title_link.text.strip()
                    relative_url = title_link.get_attribute("href")
                    
                    if relative_url:
                        if relative_url.startswith("http"):
                            job_url = relative_url
                        else:
                            job_url = "https://www.maukerja.my" + relative_url
                        
                        # Extract Job ID from URL (e.g., /en/job/21845158-motor-dispatcher)
                        if "/job/" in job_url:
                            job_id = job_url.split("/job/")[1].split("-")[0]
                except Exception as e:
                    print(f"⚠️ Error extracting title/URL: {e}")
                
                # --- Company Name ---
                try:
                    company_elem = job.find_element(By.CSS_SELECTOR, "h2#companyName" + job_id if job_id != "Not Found" else "h2[id^='companyName']")
                    company_name = company_elem.text.strip()
                except:
                    try:
                        # Alternative selector
                        company_elem = job.find_element(By.CSS_SELECTOR, "a[href*='/company/'] h2")
                        company_name = company_elem.text.strip()
                    except Exception as e:
                        print(f"⚠️ Error extracting company: {e}")
                
                # --- Location ---
                try:
                    location_elems = job.find_elements(By.CSS_SELECTOR, "p.has-text-grey-dark a.joblocation-link")
                    location = ", ".join([elem.text.strip() for elem in location_elems if elem.text.strip()])
                except Exception as e:
                    print(f"⚠️ Error extracting location: {e}")
                
                # --- Salary ---
                try:
                    salary_elem = job.find_element(By.CSS_SELECTOR, "span.has-text-primary.has-text-weight-semibold")
                    salary = salary_elem.text.strip()
                except Exception as e:
                    print(f"⚠️ Error extracting salary: {e}")
                
                # --- Job Type ---
                try:
                    job_type_elems = job.find_elements(By.CSS_SELECTOR, "a.jobtype-link")
                    job_type = ", ".join([elem.text.strip() for elem in job_type_elems if elem.text.strip()])
                except Exception as e:
                    print(f"⚠️ Error extracting job type: {e}")
                
                # --- Listing Date ---
                try:
                    # Look for "Posted X ago" text
                    date_container = job.find_element(By.CSS_SELECTOR, "div.is-italic")
                    listing_date = date_container.text.strip().replace("Posted", "").strip()
                except Exception as e:
                    print(f"⚠️ Error extracting date: {e}")
                
                # --- Skills/Categories ---
                try:
                    skill_elems = job.find_elements(By.CSS_SELECTOR, "div#job-detail-text text-base")
                    skills = [elem.text.strip() for elem in skill_elems if elem.text.strip()]
                    
                    if skills:
                        category_text = skills[0]
                        subcategory_text = ", ".join(skills[1:]) if len(skills) > 1 else "N/A"
                except Exception as e:
                    print(f"⚠️ Error extracting skills: {e}")
                
                # --- Scrape Job Description ---
                job_details = scrape_job_details(job_url) if job_url != "Not Found" else None
                
                # Combine Requirements and Responsibilities into Job Description
                job_description = "Not Found"
                if job_details:
                    desc_parts = []
                    if job_details["requirements"] != "Not Found":
                        desc_parts.append("REQUIREMENTS:\n" + job_details["requirements"])
                    if job_details["responsibilities"] != "Not Found":
                        desc_parts.append("RESPONSIBILITIES:\n" + job_details["responsibilities"])
                    
                    if desc_parts:
                        job_description = "\n\n".join(desc_parts)
                
                scraped_date = pd.to_datetime('today').strftime('%Y-%m-%d')
                
                all_jobs.append({
                    "Job ID": job_id,
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Company Type": job_details["company_type"] if job_details else "Not Found",
                    "Company Size": job_details["company_size"] if job_details else "Not Found",
                    "Industry": job_details["industry"] if job_details else "Not Found",
                    "Company SSM No": job_details["company_ssm_no"] if job_details else "Not Found",
                    "Job Vacancies": job_details["job_vacancies"] if job_details else "Not Found",
                    "Company Website": job_details["company_website"] if job_details else "Not Found",
                    "Company Social Media": job_details["company_social_media"] if job_details else "Not Found",
                    "Location": location,
                    "Working Location": job_details["working_location"] if job_details else "Not Found",
                    "Category": category_text,
                    "Subcategory": subcategory_text,
                    "Job Type": job_type,
                    "Salary": salary,
                    "Listing Date": listing_date,
                    "Posted Date": job_details["posted_date"] if job_details else "Not Found",
                    "Closing Date": job_details["closing_date"] if job_details else "Not Found",
                    "Job Description": job_description,
                    "Benefits": job_details["benefits"] if job_details else "Not Found",
                    "Skills": job_details["skills"] if job_details else "Not Found",
                    "Job URL": job_url,
                    "Scraped Date": scraped_date
                })
                
                print(f"✅ Scraped: {job_title} at {company_name}")
                
            except Exception as e:
                print(f"⚠️ Error scraping job card {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Failed to parse page {page}: {e}")
        continue

driver.quit()

# ------------------------ SAVE TO FILE ------------------------ #
df = pd.DataFrame(all_jobs)

# Add timestamp to avoid file conflicts
from datetime import datetime
timestamp = datetime.now().strftime('%d%B_%H%M%S')
excel_file = f"{timestamp}_maukerja_listings.xlsx"
csv_file = f"{timestamp}_maukerja_listings.csv"

# Save Excel file
try:
    df.to_excel(excel_file, index=False)
    print(f"✅ Excel saved: {excel_file}")
except PermissionError:
    excel_file = f"{timestamp}_maukerja_listings_v2.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"✅ Excel saved (alternate): {excel_file}")

# Save CSV file
try:
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"✅ CSV saved: {csv_file}")
except PermissionError:
    csv_file = f"{timestamp}_maukerja_listings_v2.csv"
    try:
        df.to_csv(csv_file, index=False, encoding="utf-8-sig")
        print(f"✅ CSV saved (alternate): {csv_file}")
    except:
        print(f"⚠️ Could not save CSV (file may be open). Excel file saved successfully.")

print(f"\n✅ Maukerja Scraping complete!")
print(f"📊 Total jobs scraped: {len(all_jobs)}")


🔐 MANUAL LOGIN REQUIRED
⏳ Maukerja homepage loaded.
✅ Cookies loaded from maukerja_cookies.json
🔄 Refreshing page with saved cookies...
✅ Page refreshed. Check if you're logged in.
   If not logged in, you'll need to sign in manually.

📋 INSTRUCTIONS:
   1. Check if you're already logged in (look for your profile icon)
   2. If NOT logged in:
      - Click the 'Sign In' or 'Log In' button
      - Enter your email and password
      - Complete any verification if required
   3. Once logged in, return to this console

⚠️  DO NOT close the browser window!



✋ Press ENTER after you have successfully logged in...  


✅ Cookies saved to maukerja_cookies.json

✅ Login confirmed! Starting scraping process...



Pages:   0%|                                                                                   | 0/2 [00:00<?, ?page/s]


📄 Loading Page 1 → https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page=1
🔍 Found 31 job cards on page 1


...


Page 2:  97%|██████████████████████████████████████████████████████████████████████▋  | 30/31 [03:15<00:05,  5.73s/job]

✅ Scraped: Domino'S Pizza Pantai Hill Park at Domino'S Pizza
⚠️ Company container not found
🔍 Starting skills extraction...
   Found 0 job-detail-text containers
   Trying direct flex-wrap search...
   Found 0 flex-wrap divs
   Trying section header search...
   Found Skills section header
   Found Skills section header
⚠️ No skills found after trying all methods



Pages: 100%|██████████████████████████████████████████████████████████████████████████| 2/2 [07:38<00:00, 229.07s/page]

✅ Scraped: Assistant Manager at Domino'S Pizza Kota Kemuning at Domino'S Pizza


✅ Excel saved: 03December_151424_maukerja_listings.xlsx
✅ CSV saved: 03December_151424_maukerja_listings.csv

✅ Maukerja Scraping complete!
📊 Total jobs scraped: 62


In [8]:
import time
import pandas as pd
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

# ------------------------ COOKIE MANAGEMENT ------------------------ #
COOKIE_FILE = "maukerja_cookies.json"

def save_cookies(driver, filepath=COOKIE_FILE):
    """Save cookies to a file after login."""
    with open(filepath, 'w') as f:
        json.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {filepath}")

def load_cookies(driver, filepath=COOKIE_FILE):
    """Load cookies from file to maintain login session."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            cookies = json.load(f)
            for cookie in cookies:
                if 'expiry' in cookie:
                    del cookie['expiry']
                try:
                    driver.add_cookie(cookie)
                except Exception as e:
                    print(f"⚠️ Could not add cookie: {e}")
        print(f"✅ Cookies loaded from {filepath}")
        return True
    else:
        print(f"⚠️ No cookie file found at {filepath}")
        return False

# ------------------------ SETUP ------------------------ #
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.page_load_strategy = "eager"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page={page}"

# ------------------------ MANUAL LOGIN ------------------------ #
print("\n" + "="*60)
print("🔐 MANUAL LOGIN REQUIRED")
print("="*60)

driver.get("https://www.maukerja.my/en")
print("⏳ Maukerja homepage loaded.")
time.sleep(3)

if load_cookies(driver):
    print("🔄 Refreshing page with saved cookies...")
    driver.refresh()
    time.sleep(5)
    print("✅ Page refreshed. Check if you're logged in.")
    print("   If not logged in, you'll need to sign in manually.")
else:
    print("ℹ️  No saved cookies found. You need to log in manually.")

print("\n📋 INSTRUCTIONS:")
print("   1. Check if you're already logged in (look for your profile icon)")
print("   2. If NOT logged in:")
print("      - Click the 'Sign In' or 'Log In' button")
print("      - Enter your email and password")
print("      - Complete any verification if required")
print("   3. Once logged in, return to this console")
print("\n⚠️  DO NOT close the browser window!")
print("="*60)

input("\n✋ Press ENTER after you have successfully logged in... ")

save_cookies(driver)

print("\n✅ Login confirmed! Starting scraping process...")
print("="*60 + "\n")

time.sleep(3)

# ------------------------ SAFE GET ------------------------ #
def safe_get(url, retries=3, wait_time=5):
    """Try loading a page with retries to avoid timeout crashes."""
    for i in range(retries):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"⚠️ Retry {i+1}/{retries} for {url} → {e}")
            time.sleep(wait_time)
    return False

# ------------------------ SCRAPE JOB DESCRIPTION ------------------------ #
def scrape_job_details(job_url):
    """Scrape detailed job information from the job detail page."""
    details = {
        "working_location": "Not Found",
        "requirements": "Not Found",
        "responsibilities": "Not Found",
        "benefits": "Not Found",
        "skills": "Not Found",
        "posted_date": "Not Found",
        "closing_date": "Not Found",
        "company_type": "Not Found",
        "company_size": "Not Found",
        "industry": "Not Found",
        "company_ssm_no": "Not Found",
        "job_vacancies": "Not Found",
        "company_website": "Not Found",
        "company_social_media": "Not Found"
    }
    
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])

        if not safe_get(job_url):
            return details

        time.sleep(4)  # Increased wait time for page to fully load
        
        # --- Extract Company Information using Selenium ---
        try:
            print(f"🔍 Extracting company information...")
            
            company_fields = {
                "company_type": "company type",
                "company_size": "company size",
                "industry": "industry",
                "company_ssm_no": "company ssm no",
                "job_vacancies": "job vacancies",
                "company_website": "company website"
            }
            
            company_columns = driver.find_elements(By.CSS_SELECTOR, ".has-background-grey90 .column")
            print(f"   Found {len(company_columns)} company info columns")
            
            for column in company_columns:
                try:
                    label_elem = column.find_element(By.CSS_SELECTOR, "h3.has-text-weight-bold")
                    label = label_elem.text.strip().lower()
                    
                    value_elem = column.find_element(By.TAG_NAME, "p")
                    value = value_elem.text.strip()
                    
                    for key, field_label in company_fields.items():
                        if label == field_label:
                            if key == "company_ssm_no":
                                details[key] = value if value != "-" else "Not Provided"
                            else:
                                details[key] = value
                            print(f"   ✓ {field_label.title()}: {details[key]}")
                            break
                    
                    if "website" in label and "social" not in label:
                        try:
                            link_elem = column.find_element(By.CSS_SELECTOR, "p a")
                            details["company_website"] = link_elem.get_attribute("href")
                            print(f"   ✓ Company Website: {details['company_website']}")
                        except NoSuchElementException:
                            details["company_website"] = value
                    
                    elif "social media" in label:
                        try:
                            social_links = column.find_elements(By.CSS_SELECTOR, "a[rel='nofollow']")
                            links = [link.get_attribute("href") for link in social_links if link.get_attribute("href")]
                            details["company_social_media"] = "; ".join(links) if links else "Not Found"
                            print(f"   ✓ Social Media: Found {len(links)} links")
                        except NoSuchElementException:
                            continue
                
                except NoSuchElementException:
                    continue
            
            print(f"✅ Company information extracted")
        except NoSuchElementException:
            print(f"⚠️ Company information section not found")
        except Exception as e:
            print(f"⚠️ Error extracting company information: {e}")
        
        # --- Extract Skills using Multiple Methods ---
        try:
            print(f"🔍 Extracting skills...")
            skills_list = []
            
            # Method 1: CSS Selector
            if not skills_list:
                try:
                    skill_spans = driver.find_elements(By.CSS_SELECTOR, "div.job-detail-text div.is-flex-wrap-wrap span.px-2")
                    print(f"   Method 1: Found {len(skill_spans)} skill elements")
                    for span in skill_spans:
                        skill_text = span.text.strip()
                        if skill_text:
                            skills_list.append(skill_text)
                except Exception as e:
                    print(f"   Method 1 failed: {e}")
            
            # Method 2: Broader selector with style filtering
            if not skills_list:
                try:
                    skill_spans = driver.find_elements(By.CSS_SELECTOR, "span.px-2.py-1.has-text-weight-bold")
                    print(f"   Method 2: Found {len(skill_spans)} potential elements")
                    for span in skill_spans:
                        style = span.get_attribute("style") or ""
                        if "border-radius" in style:
                            skill_text = span.text.strip()
                            if skill_text:
                                skills_list.append(skill_text)
                except Exception as e:
                    print(f"   Method 2 failed: {e}")
            
            # Method 3: XPath
            if not skills_list:
                try:
                    skill_spans = driver.find_elements(By.XPATH, "//div[contains(@class, 'job-detail-text')]//span[contains(@class, 'px-2')]")
                    print(f"   Method 3: Found {len(skill_spans)} elements via XPath")
                    for span in skill_spans:
                        skill_text = span.text.strip()
                        if skill_text and len(skill_text) > 2:
                            skills_list.append(skill_text)
                except Exception as e:
                    print(f"   Method 3 failed: {e}")
            
            # Method 4: Wait and search
            if not skills_list:
                try:
                    WebDriverWait(driver, 5).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.is-flex-wrap-wrap"))
                    )
                    skill_spans = driver.find_elements(By.CSS_SELECTOR, "div.is-flex-wrap-wrap span")
                    print(f"   Method 4: Found {len(skill_spans)} elements")
                    for span in skill_spans:
                        if "px-2" in (span.get_attribute("class") or ""):
                            skill_text = span.text.strip()
                            if skill_text:
                                skills_list.append(skill_text)
                except Exception as e:
                    print(f"   Method 4 failed: {e}")
            
            if skills_list:
                details["skills"] = ", ".join(skills_list)
                print(f"✅ Found {len(skills_list)} skills: {', '.join(skills_list[:3])}...")
            else:
                print(f"⚠️ No skills found")
                    
        except Exception as e:
            print(f"⚠️ Error extracting skills: {e}")
        
        # Use BeautifulSoup for remaining fields
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # --- Extract Working Location ---
        try:
            location_section = soup.find('div', {'id': 'job-locations'})
            if location_section:
                location_list = location_section.find('ul')
                if location_list:
                    locations = [li.get_text(strip=True) for li in location_list.find_all('li')]
                    details["working_location"] = "; ".join(locations)
        except Exception as e:
            print(f"⚠️ Error extracting working location: {e}")
        
        # --- Extract Job Description sections ---
        try:
            job_desc_section = soup.find('div', {'id': 'job-description'})
            if job_desc_section:
                sections = job_desc_section.find_all('div', class_='space-y-4')
                
                for section in sections:
                    title_elem = section.find('p', class_='font-bold')
                    if title_elem:
                        title = title_elem.get_text(strip=True).lower()
                        content_div = section.find('div', class_='job-detail-text')
                        if content_div:
                            content_parts = []
                            for ul in content_div.find_all('ul'):
                                items = [li.get_text(strip=True) for li in ul.find_all('li')]
                                content_parts.extend(items)
                            for p in content_div.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            content = "\n".join(content_parts)
                            
                            if 'requirement' in title:
                                details["requirements"] = content
                            elif 'responsibilit' in title:
                                details["responsibilities"] = content
                            elif 'benefit' in title:
                                details["benefits"] = content
        except Exception as e:
            print(f"⚠️ Error extracting job description: {e}")
        
        # --- Extract Dates ---
        try:
            date_text = soup.find('div', class_='text-base', string=lambda text: text and 'Posted' in text)
            if date_text:
                date_str = date_text.get_text(strip=True)
                if 'Posted' in date_str:
                    parts = date_str.split('•')
                    if len(parts) >= 1:
                        details["posted_date"] = parts[0].replace('Posted', '').strip()
                    if len(parts) >= 2:
                        details["closing_date"] = parts[1].replace('Closing', '').strip()
        except Exception as e:
            print(f"⚠️ Error extracting dates: {e}")

    except Exception as e:
        print(f"❌ Error fetching job details for {job_url}: {e}")
    finally:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(1)
    
    return details

# ------------------------ MAIN SCRAPER ------------------------ #
max_pages = 2  # Testing with 2 pages
all_jobs = []

for page in tqdm(range(1, max_pages + 1), desc="Pages", unit="page"):
    url = BASE_URL.format(page=page)
    print(f"\n📄 Loading Page {page} → {url}")

    if not safe_get(url):
        print(f"❌ Failed to load page {page}, skipping.")
        continue

    time.sleep(3)

    try:
        job_card_selector = "div.job-card-lite"
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector)))
        job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
        print(f"🔍 Found {len(job_cards)} job cards on page {page}")

        for idx in tqdm(range(len(job_cards)), desc=f"Page {page}", unit="job", leave=False):
            try:
                job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
                job = job_cards[idx]
                
                job_title = "Not Found"
                job_url = "Not Found"
                company_name = "Not Found"
                location = "Not Found"
                salary = "Not Found"
                job_type = "Not Found"
                listing_date = "Not Found"
                category_text = "Not Found"
                subcategory_text = "N/A"
                job_id = "Not Found"
                
                # Extract Job URL and ID
                try:
                    title_link = job.find_element(By.CSS_SELECTOR, "div.font-bold a.has-text-dark")
                    job_title = title_link.text.strip()
                    relative_url = title_link.get_attribute("href")
                    
                    if relative_url:
                        job_url = relative_url if relative_url.startswith("http") else "https://www.maukerja.my" + relative_url
                        if "/job/" in job_url:
                            job_id = job_url.split("/job/")[1].split("-")[0]
                except:
                    pass
                
                # Extract Company Name
                try:
                    company_elem = job.find_element(By.CSS_SELECTOR, "a[href*='/company/'] h2")
                    company_name = company_elem.text.strip()
                except:
                    pass
                
                # Extract Location
                try:
                    location_elems = job.find_elements(By.CSS_SELECTOR, "p.has-text-grey-dark a.joblocation-link")
                    location = ", ".join([elem.text.strip() for elem in location_elems if elem.text.strip()])
                except:
                    pass
                
                # Extract Salary
                try:
                    salary_elem = job.find_element(By.CSS_SELECTOR, "span.has-text-primary.has-text-weight-semibold")
                    salary = salary_elem.text.strip()
                except:
                    pass
                
                # Extract Job Type
                try:
                    job_type_elems = job.find_elements(By.CSS_SELECTOR, "a.jobtype-link")
                    job_type = ", ".join([elem.text.strip() for elem in job_type_elems if elem.text.strip()])
                except:
                    pass
                
                # Extract Listing Date
                try:
                    date_container = job.find_element(By.CSS_SELECTOR, "div.is-italic")
                    listing_date = date_container.text.strip().replace("Posted", "").strip()
                except:
                    pass
                
                # Extract Skills/Categories
                try:
                    skill_elems = job.find_elements(By.CSS_SELECTOR, "div#jobCardSkills a")
                    skills = [elem.text.strip() for elem in skill_elems if elem.text.strip()]
                    if skills:
                        category_text = skills[0]
                        subcategory_text = ", ".join(skills[1:]) if len(skills) > 1 else "N/A"
                except:
                    pass
                
                # Scrape Job Details
                job_details = scrape_job_details(job_url) if job_url != "Not Found" else None
                
                # Combine Requirements and Responsibilities
                job_description = "Not Found"
                if job_details:
                    desc_parts = []
                    if job_details["requirements"] != "Not Found":
                        desc_parts.append("REQUIREMENTS:\n" + job_details["requirements"])
                    if job_details["responsibilities"] != "Not Found":
                        desc_parts.append("RESPONSIBILITIES:\n" + job_details["responsibilities"])
                    if desc_parts:
                        job_description = "\n\n".join(desc_parts)
                
                scraped_date = pd.to_datetime('today').strftime('%Y-%m-%d')
                
                all_jobs.append({
                    "Job ID": job_id,
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Company Type": job_details["company_type"] if job_details else "Not Found",
                    "Company Size": job_details["company_size"] if job_details else "Not Found",
                    "Industry": job_details["industry"] if job_details else "Not Found",
                    "Company SSM No": job_details["company_ssm_no"] if job_details else "Not Found",
                    "Job Vacancies": job_details["job_vacancies"] if job_details else "Not Found",
                    "Company Website": job_details["company_website"] if job_details else "Not Found",
                    "Company Social Media": job_details["company_social_media"] if job_details else "Not Found",
                    "Location": location,
                    "Working Location": job_details["working_location"] if job_details else "Not Found",
                    "Category": category_text,
                    "Subcategory": subcategory_text,
                    "Job Type": job_type,
                    "Salary": salary,
                    "Listing Date": listing_date,
                    "Posted Date": job_details["posted_date"] if job_details else "Not Found",
                    "Closing Date": job_details["closing_date"] if job_details else "Not Found",
                    "Job Description": job_description,
                    "Benefits": job_details["benefits"] if job_details else "Not Found",
                    "Skills": job_details["skills"] if job_details else "Not Found",
                    "Job URL": job_url,
                    "Scraped Date": scraped_date
                })
                
                print(f"✅ Scraped: {job_title} at {company_name}")
                
            except Exception as e:
                print(f"⚠️ Error scraping job card {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Failed to parse page {page}: {e}")
        continue

driver.quit()

# ------------------------ SAVE TO FILE ------------------------ #
df = pd.DataFrame(all_jobs)

timestamp = datetime.now().strftime('%d%B_%H%M%S')
excel_file = f"{timestamp}_maukerja_listings.xlsx"
csv_file = f"{timestamp}_maukerja_listings.csv"

try:
    df.to_excel(excel_file, index=False)
    print(f"✅ Excel saved: {excel_file}")
except PermissionError:
    excel_file = f"{timestamp}_maukerja_listings_v2.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"✅ Excel saved (alternate): {excel_file}")

try:
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"✅ CSV saved: {csv_file}")
except PermissionError:
    csv_file = f"{timestamp}_maukerja_listings_v2.csv"
    try:
        df.to_csv(csv_file, index=False, encoding="utf-8-sig")
        print(f"✅ CSV saved (alternate): {csv_file}")
    except:
        print(f"⚠️ Could not save CSV (file may be open). Excel file saved successfully.")

print(f"\n✅ Maukerja Scraping complete!")
print(f"📊 Total jobs scraped: {len(all_jobs)}")


🔐 MANUAL LOGIN REQUIRED
⏳ Maukerja homepage loaded.
✅ Cookies loaded from maukerja_cookies.json
🔄 Refreshing page with saved cookies...
✅ Page refreshed. Check if you're logged in.
   If not logged in, you'll need to sign in manually.

📋 INSTRUCTIONS:
   1. Check if you're already logged in (look for your profile icon)
   2. If NOT logged in:
      - Click the 'Sign In' or 'Log In' button
      - Enter your email and password
      - Complete any verification if required
   3. Once logged in, return to this console

⚠️  DO NOT close the browser window!



✋ Press ENTER after you have successfully logged in...  


✅ Cookies saved to maukerja_cookies.json

✅ Login confirmed! Starting scraping process...



Pages:   0%|                                                                                   | 0/2 [00:00<?, ?page/s]


📄 Loading Page 1 → https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page=1
🔍 Found 31 job cards on page 1


...


Page 2:  97%|██████████████████████████████████████████████████████████████████████▋  | 30/31 [06:03<00:12, 12.44s/job]

✅ Scraped: Assistant Manager at Domino'S Pizza Ukay Bestari at Domino'S Pizza
🔍 Extracting company information...
   Found 0 company info columns
✅ Company information extracted
🔍 Extracting skills...
   Method 1: Found 0 skill elements
   Method 2: Found 0 potential elements
   Method 3: Found 0 elements via XPath
   Method 4 failed: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xe74103
	0xe74144
	0xc7e71d
	0xcca03d
	0xcca41b
	0xd117f2
	0xcec954
	0xd0ee17
	0xcec706
	0xcbda30
	0xcbed54
	0x10e57b4
	0x10e098a
	0xe9c392
	0xe8c4c8
	0xe9324d
	0xe7c478
	0xe7c63c
	0xe667ca
	0x76345d49
	0x7777d6db
	0x7777d661

⚠️ No skills found



Pages: 100%|██████████████████████████████████████████████████████████████████████████| 2/2 [12:40<00:00, 380.25s/page]

✅ Scraped: Assistant Manager at Domino'S Pizza Plaza Crystalville at Domino'S Pizza


✅ Excel saved: 03December_155139_maukerja_listings.xlsx
✅ CSV saved: 03December_155139_maukerja_listings.csv

✅ Maukerja Scraping complete!
📊 Total jobs scraped: 62


In [9]:
import time
import pandas as pd
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

# ------------------------ COOKIE MANAGEMENT ------------------------ #
COOKIE_FILE = "maukerja_cookies.json"

def save_cookies(driver, filepath=COOKIE_FILE):
    """Save cookies to a file after login."""
    with open(filepath, 'w') as f:
        json.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {filepath}")

def load_cookies(driver, filepath=COOKIE_FILE):
    """Load cookies from file to maintain login session."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            cookies = json.load(f)
            for cookie in cookies:
                if 'expiry' in cookie:
                    del cookie['expiry']
                try:
                    driver.add_cookie(cookie)
                except Exception as e:
                    print(f"⚠️ Could not add cookie: {e}")
        print(f"✅ Cookies loaded from {filepath}")
        return True
    else:
        print(f"⚠️ No cookie file found at {filepath}")
        return False

# ------------------------ SETUP ------------------------ #
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.page_load_strategy = "eager"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page={page}"

# ------------------------ MANUAL LOGIN ------------------------ #
print("\n" + "="*60)
print("🔐 MANUAL LOGIN REQUIRED")
print("="*60)

driver.get("https://www.maukerja.my/en")
print("⏳ Maukerja homepage loaded.")
time.sleep(3)

if load_cookies(driver):
    print("🔄 Refreshing page with saved cookies...")
    driver.refresh()
    time.sleep(5)
    print("✅ Page refreshed. Check if you're logged in.")
    print("   If not logged in, you'll need to sign in manually.")
else:
    print("ℹ️  No saved cookies found. You need to log in manually.")

print("\n📋 INSTRUCTIONS:")
print("   1. Check if you're already logged in (look for your profile icon)")
print("   2. If NOT logged in:")
print("      - Click the 'Sign In' or 'Log In' button")
print("      - Enter your email and password")
print("      - Complete any verification if required")
print("   3. Once logged in, return to this console")
print("\n⚠️  DO NOT close the browser window!")
print("="*60)

input("\n✋ Press ENTER after you have successfully logged in... ")

save_cookies(driver)

print("\n✅ Login confirmed! Starting scraping process...")
print("="*60 + "\n")

time.sleep(3)

# ------------------------ SAFE GET ------------------------ #
def safe_get(url, retries=3, wait_time=5):
    """Try loading a page with retries to avoid timeout crashes."""
    for i in range(retries):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"⚠️ Retry {i+1}/{retries} for {url} → {e}")
            time.sleep(wait_time)
    return False

# ------------------------ SCRAPE JOB DESCRIPTION ------------------------ #
def scrape_job_details(job_url, job_id):
    """Scrape detailed job information from the job detail page."""
    details = {
        "working_location": "Not Found",
        "requirements": "Not Found",
        "responsibilities": "Not Found",
        "benefits": "Not Found",
        "skills": "Not Found",
        "posted_date": "Not Found",
        "closing_date": "Not Found",
        "company_type": "Not Found",
        "company_size": "Not Found",
        "industry": "Not Found",
        "company_ssm_no": "Not Found",
        "job_vacancies": "Not Found",
        "company_website": "Not Found",
        "company_social_media": "Not Found"
    }
    
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])

        if not safe_get(job_url):
            return details

        time.sleep(5)  # Increased wait time
        
        # Scroll to load all content
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # --- Extract Company Information using Selenium ---
        try:
            print(f"🔍 Extracting company information...")
            
            company_fields = {
                "company_type": "company type",
                "company_size": "company size",
                "industry": "industry",
                "company_ssm_no": "company ssm no",
                "job_vacancies": "job vacancies",
                "company_website": "company website"
            }
            
            company_columns = driver.find_elements(By.CSS_SELECTOR, ".has-background-grey90 .column")
            print(f"   Found {len(company_columns)} company info columns")
            
            for column in company_columns:
                try:
                    label_elem = column.find_element(By.CSS_SELECTOR, "h3.has-text-weight-bold")
                    label = label_elem.text.strip().lower()
                    
                    value_elem = column.find_element(By.TAG_NAME, "p")
                    value = value_elem.text.strip()
                    
                    for key, field_label in company_fields.items():
                        if label == field_label:
                            if key == "company_ssm_no":
                                details[key] = value if value != "-" else "Not Provided"
                            else:
                                details[key] = value
                            print(f"   ✓ {field_label.title()}: {details[key]}")
                            break
                    
                    if "website" in label and "social" not in label:
                        try:
                            link_elem = column.find_element(By.CSS_SELECTOR, "p a")
                            details["company_website"] = link_elem.get_attribute("href")
                            print(f"   ✓ Company Website: {details['company_website']}")
                        except NoSuchElementException:
                            details["company_website"] = value
                    
                    elif "social media" in label:
                        try:
                            social_links = column.find_elements(By.CSS_SELECTOR, "a[rel='nofollow']")
                            links = [link.get_attribute("href") for link in social_links if link.get_attribute("href")]
                            details["company_social_media"] = "; ".join(links) if links else "Not Found"
                            print(f"   ✓ Social Media: Found {len(links)} links")
                        except NoSuchElementException:
                            continue
                
                except NoSuchElementException:
                    continue
            
            print(f"✅ Company information extracted")
        except NoSuchElementException:
            print(f"⚠️ Company information section not found")
        except Exception as e:
            print(f"⚠️ Error extracting company information: {e}")
        
        # --- Extract Skills with Enhanced Debugging ---
        try:
            print(f"\n🔍 Extracting skills...")
            print(f"   Current URL: {driver.current_url}")
            skills_list = []
            
            # Debug: Check page content
            all_spans = driver.find_elements(By.TAG_NAME, "span")
            print(f"   Total spans on page: {len(all_spans)}")
            
            # Check if Skills section exists
            try:
                driver.find_element(By.XPATH, "//*[contains(text(), 'Skills') or contains(text(), 'skills')]")
                print(f"   ✓ 'Skills' text found on page")
            except:
                print(f"   ✗ 'Skills' text NOT found on page")
            
            # Method 1: Direct CSS - job-detail-text
            print(f"\n   === Method 1: job-detail-text CSS ===")
            try:
                skill_spans = driver.find_elements(By.CSS_SELECTOR, "div.job-detail-text span.px-2")
                print(f"   Found {len(skill_spans)} elements")
                if len(skill_spans) > 0:
                    print(f"   Sample: text='{skill_spans[0].text[:30]}', class='{skill_spans[0].get_attribute('class')[:50]}'")
                
                for span in skill_spans:
                    text = span.text.strip()
                    if text and len(text) > 1:
                        skills_list.append(text)
                        print(f"   ✓ {text}")
            except Exception as e:
                print(f"   Error: {e}")
            
            # Method 2: All px-2 spans with filtering
            if not skills_list:
                print(f"\n   === Method 2: All px-2 spans ===")
                try:
                    all_px2 = driver.find_elements(By.CSS_SELECTOR, "span.px-2")
                    print(f"   Found {len(all_px2)} px-2 spans")
                    
                    for i, span in enumerate(all_px2):
                        text = span.text.strip()
                        classes = span.get_attribute("class")
                        
                        if i < 5:  # Show first 5 for debugging
                            print(f"   [{i}] text='{text[:40]}' | class='{classes[:60]}'")
                        
                        if "py-1" in classes and "has-text-weight-bold" in classes:
                            if text and len(text) > 1 and len(text) < 50:  # Skills are typically short
                                skills_list.append(text)
                                print(f"   ✓ Added: {text}")
                except Exception as e:
                    print(f"   Error: {e}")
            
            # Method 3: flex-wrap-wrap container
            if not skills_list:
                print(f"\n   === Method 3: flex-wrap-wrap ===")
                try:
                    containers = driver.find_elements(By.CSS_SELECTOR, "div.is-flex-wrap-wrap")
                    print(f"   Found {len(containers)} flex-wrap containers")
                    
                    for idx, container in enumerate(containers):
                        preview = container.text[:80].replace('\n', ' ')
                        print(f"   Container {idx}: '{preview}'")
                        
                        spans = container.find_elements(By.TAG_NAME, "span")
                        for span in spans:
                            classes = span.get_attribute("class") or ""
                            text = span.text.strip()
                            if "px-2" in classes and text and len(text) > 1:
                                skills_list.append(text)
                                print(f"   ✓ Added: {text}")
                except Exception as e:
                    print(f"   Error: {e}")
            
            # Method 4: XPath
            if not skills_list:
                print(f"\n   === Method 4: XPath ===")
                try:
                    spans = driver.find_elements(By.XPATH, 
                        "//div[contains(@class, 'is-flex-wrap-wrap')]//span[contains(@class, 'px-2')]")
                    print(f"   Found {len(spans)} elements")
                    
                    for span in spans:
                        text = span.text.strip()
                        if text and len(text) > 1:
                            skills_list.append(text)
                            print(f"   ✓ Added: {text}")
                except Exception as e:
                    print(f"   Error: {e}")
            
            # Save debug file if nothing found
            if not skills_list:
                print(f"\n   ❌ NO SKILLS FOUND - Saving debug file")
                debug_file = f"debug_page_{job_id}.html"
                with open(debug_file, "w", encoding="utf-8") as f:
                    f.write(driver.page_source)
                print(f"   Saved: {debug_file}")
                
                # Check page source
                source = driver.page_source
                checks = {
                    "job-detail-text": "job-detail-text" in source,
                    "is-flex-wrap-wrap": "is-flex-wrap-wrap" in source,
                    "px-2": "px-2" in source,
                    "Skills": "Skills" in source or "skills" in source,
                    "Administrative": "Administrative" in source
                }
                print(f"   Page checks:")
                for name, found in checks.items():
                    print(f"   {'✓' if found else '✗'} {name}: {found}")
            
            if skills_list:
                details["skills"] = ", ".join(skills_list)
                print(f"\n✅ Found {len(skills_list)} skills")
            else:
                print(f"\n❌ No skills extracted")
                    
        except Exception as e:
            print(f"⚠️ Critical error: {e}")
            import traceback
            traceback.print_exc()
        
        # Use BeautifulSoup for remaining fields
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # --- Extract Working Location ---
        try:
            location_section = soup.find('div', {'id': 'job-locations'})
            if location_section:
                location_list = location_section.find('ul')
                if location_list:
                    locations = [li.get_text(strip=True) for li in location_list.find_all('li')]
                    details["working_location"] = "; ".join(locations)
        except Exception as e:
            print(f"⚠️ Error extracting location: {e}")
        
        # --- Extract Job Description sections ---
        try:
            job_desc_section = soup.find('div', {'id': 'job-description'})
            if job_desc_section:
                sections = job_desc_section.find_all('div', class_='space-y-4')
                
                for section in sections:
                    title_elem = section.find('p', class_='font-bold')
                    if title_elem:
                        title = title_elem.get_text(strip=True).lower()
                        content_div = section.find('div', class_='job-detail-text')
                        if content_div:
                            content_parts = []
                            for ul in content_div.find_all('ul'):
                                items = [li.get_text(strip=True) for li in ul.find_all('li')]
                                content_parts.extend(items)
                            for p in content_div.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            content = "\n".join(content_parts)
                            
                            if 'requirement' in title:
                                details["requirements"] = content
                            elif 'responsibilit' in title:
                                details["responsibilities"] = content
                            elif 'benefit' in title:
                                details["benefits"] = content
        except Exception as e:
            print(f"⚠️ Error extracting descriptions: {e}")
        
        # --- Extract Dates ---
        try:
            date_text = soup.find('div', class_='text-base', string=lambda text: text and 'Posted' in text)
            if date_text:
                date_str = date_text.get_text(strip=True)
                if 'Posted' in date_str:
                    parts = date_str.split('•')
                    if len(parts) >= 1:
                        details["posted_date"] = parts[0].replace('Posted', '').strip()
                    if len(parts) >= 2:
                        details["closing_date"] = parts[1].replace('Closing', '').strip()
        except Exception as e:
            print(f"⚠️ Error extracting dates: {e}")

    except Exception as e:
        print(f"❌ Error fetching details: {e}")
    finally:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(1)
    
    return details

# ------------------------ MAIN SCRAPER ------------------------ #
max_pages = 2
all_jobs = []

for page in tqdm(range(1, max_pages + 1), desc="Pages", unit="page"):
    url = BASE_URL.format(page=page)
    print(f"\n📄 Loading Page {page} → {url}")

    if not safe_get(url):
        print(f"❌ Failed to load page {page}")
        continue

    time.sleep(3)

    try:
        job_card_selector = "div.job-card-lite"
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector)))
        job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
        print(f"🔍 Found {len(job_cards)} jobs")

        for idx in tqdm(range(len(job_cards)), desc=f"Page {page}", unit="job", leave=False):
            try:
                job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
                job = job_cards[idx]
                
                job_title = job_url = company_name = location = salary = "Not Found"
                job_type = listing_date = category_text = job_id = "Not Found"
                subcategory_text = "N/A"
                
                try:
                    title_link = job.find_element(By.CSS_SELECTOR, "div.font-bold a.has-text-dark")
                    job_title = title_link.text.strip()
                    relative_url = title_link.get_attribute("href")
                    
                    if relative_url:
                        job_url = relative_url if relative_url.startswith("http") else "https://www.maukerja.my" + relative_url
                        if "/job/" in job_url:
                            job_id = job_url.split("/job/")[1].split("-")[0]
                except:
                    pass
                
                try:
                    company_elem = job.find_element(By.CSS_SELECTOR, "a[href*='/company/'] h2")
                    company_name = company_elem.text.strip()
                except:
                    pass
                
                try:
                    location_elems = job.find_elements(By.CSS_SELECTOR, "p.has-text-grey-dark a.joblocation-link")
                    location = ", ".join([e.text.strip() for e in location_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    salary_elem = job.find_element(By.CSS_SELECTOR, "span.has-text-primary.has-text-weight-semibold")
                    salary = salary_elem.text.strip()
                except:
                    pass
                
                try:
                    job_type_elems = job.find_elements(By.CSS_SELECTOR, "a.jobtype-link")
                    job_type = ", ".join([e.text.strip() for e in job_type_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    date_container = job.find_element(By.CSS_SELECTOR, "div.is-italic")
                    listing_date = date_container.text.strip().replace("Posted", "").strip()
                except:
                    pass
                
                try:
                    skill_elems = job.find_elements(By.CSS_SELECTOR, "div#jobCardSkills a")
                    skills = [e.text.strip() for e in skill_elems if e.text.strip()]
                    if skills:
                        category_text = skills[0]
                        subcategory_text = ", ".join(skills[1:]) if len(skills) > 1 else "N/A"
                except:
                    pass
                
                job_details = scrape_job_details(job_url, job_id) if job_url != "Not Found" else None
                
                job_description = "Not Found"
                if job_details:
                    parts = []
                    if job_details["requirements"] != "Not Found":
                        parts.append("REQUIREMENTS:\n" + job_details["requirements"])
                    if job_details["responsibilities"] != "Not Found":
                        parts.append("RESPONSIBILITIES:\n" + job_details["responsibilities"])
                    if parts:
                        job_description = "\n\n".join(parts)
                
                scraped_date = pd.to_datetime('today').strftime('%Y-%m-%d')
                
                all_jobs.append({
                    "Job ID": job_id,
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Company Type": job_details["company_type"] if job_details else "Not Found",
                    "Company Size": job_details["company_size"] if job_details else "Not Found",
                    "Industry": job_details["industry"] if job_details else "Not Found",
                    "Company SSM No": job_details["company_ssm_no"] if job_details else "Not Found",
                    "Job Vacancies": job_details["job_vacancies"] if job_details else "Not Found",
                    "Company Website": job_details["company_website"] if job_details else "Not Found",
                    "Company Social Media": job_details["company_social_media"] if job_details else "Not Found",
                    "Location": location,
                    "Working Location": job_details["working_location"] if job_details else "Not Found",
                    "Category": category_text,
                    "Subcategory": subcategory_text,
                    "Job Type": job_type,
                    "Salary": salary,
                    "Listing Date": listing_date,
                    "Posted Date": job_details["posted_date"] if job_details else "Not Found",
                    "Closing Date": job_details["closing_date"] if job_details else "Not Found",
                    "Job Description": job_description,
                    "Benefits": job_details["benefits"] if job_details else "Not Found",
                    "Skills": job_details["skills"] if job_details else "Not Found",
                    "Job URL": job_url,
                    "Scraped Date": scraped_date
                })
                
                print(f"✅ {job_title} @ {company_name}")
                
            except Exception as e:
                print(f"⚠️ Error on job {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Page {page} error: {e}")
        continue

driver.quit()

# ------------------------ SAVE FILES ------------------------ #
df = pd.DataFrame(all_jobs)

timestamp = datetime.now().strftime('%d%B_%H%M%S')
excel_file = f"{timestamp}_maukerja_listings.xlsx"
csv_file = f"{timestamp}_maukerja_listings.csv"

try:
    df.to_excel(excel_file, index=False)
    print(f"✅ Excel: {excel_file}")
except PermissionError:
    excel_file = f"{timestamp}_maukerja_v2.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"✅ Excel: {excel_file}")

try:
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"✅ CSV: {csv_file}")
except PermissionError:
    csv_file = f"{timestamp}_maukerja_v2.csv"
    try:
        df.to_csv(csv_file, index=False, encoding="utf-8-sig")
        print(f"✅ CSV: {csv_file}")
    except:
        print(f"⚠️ CSV failed (file open?)")

print(f"\n✅ Complete! Scraped {len(all_jobs)} jobs")


🔐 MANUAL LOGIN REQUIRED
⏳ Maukerja homepage loaded.
✅ Cookies loaded from maukerja_cookies.json
🔄 Refreshing page with saved cookies...
✅ Page refreshed. Check if you're logged in.
   If not logged in, you'll need to sign in manually.

📋 INSTRUCTIONS:
   1. Check if you're already logged in (look for your profile icon)
   2. If NOT logged in:
      - Click the 'Sign In' or 'Log In' button
      - Enter your email and password
      - Complete any verification if required
   3. Once logged in, return to this console

⚠️  DO NOT close the browser window!



✋ Press ENTER after you have successfully logged in...  


✅ Cookies saved to maukerja_cookies.json

✅ Login confirmed! Starting scraping process...



Pages:   0%|                                                                                   | 0/2 [00:00<?, ?page/s]


📄 Loading Page 1 → https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page=1
🔍 Found 31 jobs


...


Page 2:  97%|██████████████████████████████████████████████████████████████████████▋  | 30/31 [05:23<00:11, 11.12s/job]

✅ Assistant Manager at Domino'S Pizza Pandan Jaya @ Domino'S Pizza
🔍 Extracting company information...
   Found 7 company info columns
   ✓ Company Type: Multinational
   ✓ Company Size: >500
   ✓ Industry: Food / Beverage
   ✓ Company Ssm No: 419060-A
   ✓ Job Vacancies: 18741
   ✓ Company Website: http://www.dominos.com.my
   ✓ Company Website: http://www.dominos.com.my/
✅ Company information extracted

🔍 Extracting skills...
   Current URL: https://www.maukerja.my/en/job/25086555-assistant-manager-at-dominos-pizza-taman-midah
   Total spans on page: 210
   ✓ 'Skills' text found on page

   === Method 1: job-detail-text CSS ===
   Found 0 elements

   === Method 2: All px-2 spans ===
   Found 1 px-2 spans
   [0] text='MRT' | class='snap-start shrink-0 px-2 py-1 rounded-md bg-white border-sol'

   === Method 3: flex-wrap-wrap ===
   Found 0 flex-wrap containers

   === Method 4: XPath ===
   Found 0 elements

   ❌ NO SKILLS FOUND - Saving debug file
   Saved: debug_page_25086555.html



Pages: 100%|██████████████████████████████████████████████████████████████████████████| 2/2 [11:41<00:00, 350.70s/page]

✅ Assistant Manager At Domino'S Pizza Taman Midah @ Domino'S Pizza


✅ Excel: 03December_162717_maukerja_listings.xlsx
✅ CSV: 03December_162717_maukerja_listings.csv

✅ Complete! Scraped 62 jobs


In [10]:
import time
import pandas as pd
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

# ------------------------ COOKIE MANAGEMENT ------------------------ #
COOKIE_FILE = "maukerja_cookies.json"

def save_cookies(driver, filepath=COOKIE_FILE):
    """Save cookies to a file after login."""
    with open(filepath, 'w') as f:
        json.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {filepath}")

def load_cookies(driver, filepath=COOKIE_FILE):
    """Load cookies from file to maintain login session."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            cookies = json.load(f)
            for cookie in cookies:
                if 'expiry' in cookie:
                    del cookie['expiry']
                try:
                    driver.add_cookie(cookie)
                except Exception as e:
                    print(f"⚠️ Could not add cookie: {e}")
        print(f"✅ Cookies loaded from {filepath}")
        return True
    else:
        print(f"⚠️ No cookie file found at {filepath}")
        return False

# ------------------------ SETUP ------------------------ #
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.page_load_strategy = "eager"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page={page}"

# ------------------------ MANUAL LOGIN ------------------------ #
print("\n" + "="*60)
print("🔐 MANUAL LOGIN REQUIRED")
print("="*60)

driver.get("https://www.maukerja.my/en")
print("⏳ Maukerja homepage loaded.")
time.sleep(3)

if load_cookies(driver):
    print("🔄 Refreshing page with saved cookies...")
    driver.refresh()
    time.sleep(5)
    print("✅ Page refreshed. Check if you're logged in.")
else:
    print("ℹ️  No saved cookies found. You need to log in manually.")

print("\n📋 INSTRUCTIONS:")
print("   1. Check if you're already logged in")
print("   2. If NOT logged in: Click 'Sign In' and enter credentials")
print("   3. Once logged in, return to this console")
print("\n⚠️  DO NOT close the browser window!")
print("="*60)

input("\n✋ Press ENTER after you have successfully logged in... ")

save_cookies(driver)
print("\n✅ Login confirmed! Starting scraping process...")
print("="*60 + "\n")
time.sleep(3)

# ------------------------ SAFE GET ------------------------ #
def safe_get(url, retries=3, wait_time=5):
    """Try loading a page with retries."""
    for i in range(retries):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"⚠️ Retry {i+1}/{retries} → {e}")
            time.sleep(wait_time)
    return False

# ------------------------ SCRAPE JOB DETAILS ------------------------ #
def scrape_job_details(job_url, job_id):
    """Scrape detailed job information."""
    details = {
        "working_location": "Not Found",
        "requirements": "Not Found",
        "responsibilities": "Not Found",
        "benefits": "Not Found",
        "skills": "Not Found",
        "posted_date": "Not Found",
        "closing_date": "Not Found",
        "company_type": "Not Found",
        "company_size": "Not Found",
        "industry": "Not Found",
        "company_ssm_no": "Not Found",
        "job_vacancies": "Not Found",
        "company_website": "Not Found",
        "company_social_media": "Not Found"
    }
    
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])

        if not safe_get(job_url):
            return details

        time.sleep(5)
        
        # Scroll to load all content
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # Get page source for BeautifulSoup
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # --- Extract Company Information ---
        try:
            print(f"🔍 Company info...")
            
            company_fields = {
                "company_type": "company type",
                "company_size": "company size",
                "industry": "industry",
                "company_ssm_no": "company ssm no",
                "job_vacancies": "job vacancies",
                "company_website": "company website"
            }
            
            company_columns = driver.find_elements(By.CSS_SELECTOR, ".has-background-grey90 .column")
            
            for column in company_columns:
                try:
                    label_elem = column.find_element(By.CSS_SELECTOR, "h3.has-text-weight-bold")
                    label = label_elem.text.strip().lower()
                    
                    value_elem = column.find_element(By.TAG_NAME, "p")
                    value = value_elem.text.strip()
                    
                    for key, field_label in company_fields.items():
                        if label == field_label:
                            if key == "company_ssm_no":
                                details[key] = value if value != "-" else "Not Provided"
                            else:
                                details[key] = value
                            break
                    
                    if "website" in label and "social" not in label:
                        try:
                            link_elem = column.find_element(By.CSS_SELECTOR, "p a")
                            details["company_website"] = link_elem.get_attribute("href")
                        except NoSuchElementException:
                            details["company_website"] = value
                    
                    elif "social media" in label:
                        try:
                            social_links = column.find_elements(By.CSS_SELECTOR, "a[rel='nofollow']")
                            links = [link.get_attribute("href") for link in social_links if link.get_attribute("href")]
                            details["company_social_media"] = "; ".join(links) if links else "Not Found"
                        except NoSuchElementException:
                            continue
                
                except NoSuchElementException:
                    continue
            
            print(f"   ✓ Company info extracted")
        except Exception as e:
            print(f"   ⚠️ Company info error: {e}")
        
        # --- Extract Skills using BeautifulSoup ---
        try:
            print(f"🔍 Skills...")
            skills_list = []
            
            # Method 1: Find is-flex-wrap-wrap containers
            flex_containers = soup.find_all('div', class_='is-flex-wrap-wrap')
            print(f"   Found {len(flex_containers)} flex containers")
            
            for container in flex_containers:
                spans = container.find_all('span', class_='px-2')
                
                for span in spans:
                    classes = span.get('class', [])
                    if 'py-1' in classes and 'has-text-weight-bold' in classes:
                        text = span.get_text(strip=True)
                        if text and 1 < len(text) < 50:
                            skills_list.append(text)
            
            # Method 2: Find by class combination
            if not skills_list:
                print(f"   Trying method 2...")
                all_spans = soup.find_all('span', class_='px-2 py-1 has-text-weight-bold')
                
                for span in all_spans:
                    style = span.get('style', '')
                    if 'border-radius' in style or 'border:' in style:
                        text = span.get_text(strip=True)
                        if text and 1 < len(text) < 50:
                            skills_list.append(text)
            
            # Method 3: Check all spans with specific criteria
            if not skills_list:
                print(f"   Trying method 3...")
                all_spans = soup.find_all('span')
                
                for span in all_spans:
                    classes = span.get('class', [])
                    style = span.get('style', '')
                    
                    if ('px-2' in classes and 'py-1' in classes and 
                        'has-text-weight-bold' in classes and 'border-radius' in style):
                        text = span.get_text(strip=True)
                        if text and 1 < len(text) < 50:
                            skills_list.append(text)
            
            if skills_list:
                details["skills"] = ", ".join(skills_list)
                print(f"   ✓ Found {len(skills_list)} skills: {', '.join(skills_list[:3])}...")
            else:
                print(f"   ⚠️ No skills found")
                    
        except Exception as e:
            print(f"   ⚠️ Skills error: {e}")
        
        # --- Extract Working Location ---
        try:
            location_section = soup.find('div', {'id': 'job-locations'})
            if location_section:
                location_list = location_section.find('ul')
                if location_list:
                    locations = [li.get_text(strip=True) for li in location_list.find_all('li')]
                    details["working_location"] = "; ".join(locations)
        except Exception as e:
            pass
        
        # --- Extract Job Description sections ---
        try:
            job_desc_section = soup.find('div', {'id': 'job-description'})
            if job_desc_section:
                sections = job_desc_section.find_all('div', class_='space-y-4')
                
                for section in sections:
                    title_elem = section.find('p', class_='font-bold')
                    if title_elem:
                        title = title_elem.get_text(strip=True).lower()
                        content_div = section.find('div', class_='job-detail-text')
                        if content_div:
                            content_parts = []
                            for ul in content_div.find_all('ul'):
                                items = [li.get_text(strip=True) for li in ul.find_all('li')]
                                content_parts.extend(items)
                            for p in content_div.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            content = "\n".join(content_parts)
                            
                            if 'requirement' in title:
                                details["requirements"] = content
                            elif 'responsibilit' in title:
                                details["responsibilities"] = content
                            elif 'benefit' in title:
                                details["benefits"] = content
        except Exception as e:
            pass
        
        # --- Extract Dates ---
        try:
            date_text = soup.find('div', class_='text-base', string=lambda text: text and 'Posted' in text)
            if date_text:
                date_str = date_text.get_text(strip=True)
                if 'Posted' in date_str:
                    parts = date_str.split('•')
                    if len(parts) >= 1:
                        details["posted_date"] = parts[0].replace('Posted', '').strip()
                    if len(parts) >= 2:
                        details["closing_date"] = parts[1].replace('Closing', '').strip()
        except Exception as e:
            pass

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(1)
    
    return details

# ------------------------ MAIN SCRAPER ------------------------ #
max_pages = 2
all_jobs = []

for page in tqdm(range(1, max_pages + 1), desc="Pages", unit="page"):
    url = BASE_URL.format(page=page)
    print(f"\n📄 Page {page}")

    if not safe_get(url):
        print(f"❌ Failed to load page {page}")
        continue

    time.sleep(3)

    try:
        job_card_selector = "div.job-card-lite"
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector)))
        job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
        print(f"🔍 Found {len(job_cards)} jobs")

        for idx in tqdm(range(len(job_cards)), desc=f"Page {page}", unit="job", leave=False):
            try:
                job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
                job = job_cards[idx]
                
                job_title = job_url = company_name = location = salary = "Not Found"
                job_type = listing_date = category_text = job_id = "Not Found"
                subcategory_text = "N/A"
                
                try:
                    title_link = job.find_element(By.CSS_SELECTOR, "div.font-bold a.has-text-dark")
                    job_title = title_link.text.strip()
                    relative_url = title_link.get_attribute("href")
                    
                    if relative_url:
                        job_url = relative_url if relative_url.startswith("http") else "https://www.maukerja.my" + relative_url
                        if "/job/" in job_url:
                            job_id = job_url.split("/job/")[1].split("-")[0]
                except:
                    pass
                
                try:
                    company_elem = job.find_element(By.CSS_SELECTOR, "a[href*='/company/'] h2")
                    company_name = company_elem.text.strip()
                except:
                    pass
                
                try:
                    location_elems = job.find_elements(By.CSS_SELECTOR, "p.has-text-grey-dark a.joblocation-link")
                    location = ", ".join([e.text.strip() for e in location_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    salary_elem = job.find_element(By.CSS_SELECTOR, "span.has-text-primary.has-text-weight-semibold")
                    salary = salary_elem.text.strip()
                except:
                    pass
                
                try:
                    job_type_elems = job.find_elements(By.CSS_SELECTOR, "a.jobtype-link")
                    job_type = ", ".join([e.text.strip() for e in job_type_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    date_container = job.find_element(By.CSS_SELECTOR, "div.is-italic")
                    listing_date = date_container.text.strip().replace("Posted", "").strip()
                except:
                    pass
                
                try:
                    skill_elems = job.find_elements(By.CSS_SELECTOR, "div#jobCardSkills a")
                    skills = [e.text.strip() for e in skill_elems if e.text.strip()]
                    if skills:
                        category_text = skills[0]
                        subcategory_text = ", ".join(skills[1:]) if len(skills) > 1 else "N/A"
                except:
                    pass
                
                job_details = scrape_job_details(job_url, job_id) if job_url != "Not Found" else None
                
                job_description = "Not Found"
                if job_details:
                    parts = []
                    if job_details["requirements"] != "Not Found":
                        parts.append("REQUIREMENTS:\n" + job_details["requirements"])
                    if job_details["responsibilities"] != "Not Found":
                        parts.append("RESPONSIBILITIES:\n" + job_details["responsibilities"])
                    if parts:
                        job_description = "\n\n".join(parts)
                
                scraped_date = pd.to_datetime('today').strftime('%Y-%m-%d')
                
                all_jobs.append({
                    "Job ID": job_id,
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Company Type": job_details["company_type"] if job_details else "Not Found",
                    "Company Size": job_details["company_size"] if job_details else "Not Found",
                    "Industry": job_details["industry"] if job_details else "Not Found",
                    "Company SSM No": job_details["company_ssm_no"] if job_details else "Not Found",
                    "Job Vacancies": job_details["job_vacancies"] if job_details else "Not Found",
                    "Company Website": job_details["company_website"] if job_details else "Not Found",
                    "Company Social Media": job_details["company_social_media"] if job_details else "Not Found",
                    "Location": location,
                    "Working Location": job_details["working_location"] if job_details else "Not Found",
                    "Category": category_text,
                    "Subcategory": subcategory_text,
                    "Job Type": job_type,
                    "Salary": salary,
                    "Listing Date": listing_date,
                    "Posted Date": job_details["posted_date"] if job_details else "Not Found",
                    "Closing Date": job_details["closing_date"] if job_details else "Not Found",
                    "Job Description": job_description,
                    "Benefits": job_details["benefits"] if job_details else "Not Found",
                    "Skills": job_details["skills"] if job_details else "Not Found",
                    "Job URL": job_url,
                    "Scraped Date": scraped_date
                })
                
                print(f"✅ {job_title[:40]} @ {company_name[:30]}")
                
            except Exception as e:
                print(f"⚠️ Job {idx} error: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Page {page} error: {e}")
        continue

driver.quit()

# ------------------------ SAVE FILES ------------------------ #
df = pd.DataFrame(all_jobs)

timestamp = datetime.now().strftime('%d%B_%H%M%S')
excel_file = f"{timestamp}_maukerja_listings.xlsx"
csv_file = f"{timestamp}_maukerja_listings.csv"

try:
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")
except PermissionError:
    excel_file = f"{timestamp}_maukerja_v2.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")

try:
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"✅ CSV: {csv_file}")
except PermissionError:
    csv_file = f"{timestamp}_maukerja_v2.csv"
    try:
        df.to_csv(csv_file, index=False, encoding="utf-8-sig")
        print(f"✅ CSV: {csv_file}")
    except:
        print(f"⚠️ CSV failed")

print(f"\n🎉 Complete! Scraped {len(all_jobs)} jobs total")


🔐 MANUAL LOGIN REQUIRED
⏳ Maukerja homepage loaded.
✅ Cookies loaded from maukerja_cookies.json
🔄 Refreshing page with saved cookies...
✅ Page refreshed. Check if you're logged in.

📋 INSTRUCTIONS:
   1. Check if you're already logged in
   2. If NOT logged in: Click 'Sign In' and enter credentials
   3. Once logged in, return to this console

⚠️  DO NOT close the browser window!



✋ Press ENTER after you have successfully logged in...  


✅ Cookies saved to maukerja_cookies.json

✅ Login confirmed! Starting scraping process...



Pages:   0%|                                                                                   | 0/2 [00:00<?, ?page/s]


📄 Page 1
🔍 Found 31 jobs


...


Page 2:  97%|██████████████████████████████████████████████████████████████████████▋  | 30/31 [05:36<00:11, 11.61s/job]

✅ Airbnb Front Office Executive @ Guestonic Property Management 
🔍 Company info...
   ✓ Company info extracted
🔍 Skills...
   Found 0 flex containers
   Trying method 2...
   Trying method 3...
   ⚠️ No skills found



Pages: 100%|██████████████████████████████████████████████████████████████████████████| 2/2 [11:33<00:00, 346.88s/page]

✅ Business Development Executive @ Katch International Sdn Bhd



✅ Excel: 04December_102134_maukerja_listings.xlsx
✅ CSV: 04December_102134_maukerja_listings.csv

🎉 Complete! Scraped 62 jobs total


In [11]:
import time
import pandas as pd
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

# ------------------------ COOKIE MANAGEMENT ------------------------ #
COOKIE_FILE = "maukerja_cookies.json"

def save_cookies(driver, filepath=COOKIE_FILE):
    """Save cookies to a file after login."""
    with open(filepath, 'w') as f:
        json.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {filepath}")

def load_cookies(driver, filepath=COOKIE_FILE):
    """Load cookies from file to maintain login session."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            cookies = json.load(f)
            for cookie in cookies:
                if 'expiry' in cookie:
                    del cookie['expiry']
                try:
                    driver.add_cookie(cookie)
                except Exception as e:
                    print(f"⚠️ Could not add cookie: {e}")
        print(f"✅ Cookies loaded from {filepath}")
        return True
    else:
        print(f"⚠️ No cookie file found at {filepath}")
        return False

# ------------------------ SETUP ------------------------ #
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.page_load_strategy = "eager"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page={page}"

# ------------------------ MANUAL LOGIN ------------------------ #
print("\n" + "="*60)
print("🔐 MANUAL LOGIN REQUIRED")
print("="*60)

driver.get("https://www.maukerja.my/en")
print("⏳ Maukerja homepage loaded.")
time.sleep(3)

if load_cookies(driver):
    print("🔄 Refreshing page with saved cookies...")
    driver.refresh()
    time.sleep(5)
    print("✅ Page refreshed. Check if you're logged in.")
else:
    print("ℹ️  No saved cookies found. You need to log in manually.")

print("\n📋 INSTRUCTIONS:")
print("   1. Check if you're already logged in")
print("   2. If NOT logged in: Click 'Sign In' and enter credentials")
print("   3. Once logged in, return to this console")
print("\n⚠️  DO NOT close the browser window!")
print("="*60)

input("\n✋ Press ENTER after you have successfully logged in... ")

save_cookies(driver)
print("\n✅ Login confirmed! Starting scraping process...")
print("="*60 + "\n")
time.sleep(3)

# ------------------------ SAFE GET ------------------------ #
def safe_get(url, retries=3, wait_time=5):
    """Try loading a page with retries."""
    for i in range(retries):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"⚠️ Retry {i+1}/{retries} → {e}")
            time.sleep(wait_time)
    return False

# ------------------------ SCRAPE JOB DETAILS ------------------------ #
def scrape_job_details(job_url, job_id):
    """Scrape detailed job information."""
    details = {
        "working_location": "Not Found",
        "requirements": "Not Found",
        "responsibilities": "Not Found",
        "benefits": "Not Found",
        "skills": "Not Found",
        "posted_date": "Not Found",
        "closing_date": "Not Found",
        "company_type": "Not Found",
        "company_size": "Not Found",
        "industry": "Not Found",
        "company_ssm_no": "Not Found",
        "job_vacancies": "Not Found",
        "company_website": "Not Found",
        "company_social_media": "Not Found"
    }
    
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])

        if not safe_get(job_url):
            return details

        time.sleep(5)
        
        # Scroll to load all content
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # Get page source for BeautifulSoup
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # --- Extract Company Information ---
        try:
            print(f"🔍 Company info...")
            
            company_fields = {
                "company_type": "company type",
                "company_size": "company size",
                "industry": "industry",
                "company_ssm_no": "company ssm no",
                "job_vacancies": "job vacancies",
                "company_website": "company website"
            }
            
            company_columns = driver.find_elements(By.CSS_SELECTOR, ".has-background-grey90 .column")
            
            for column in company_columns:
                try:
                    label_elem = column.find_element(By.CSS_SELECTOR, "h3.has-text-weight-bold")
                    label = label_elem.text.strip().lower()
                    
                    value_elem = column.find_element(By.TAG_NAME, "p")
                    value = value_elem.text.strip()
                    
                    for key, field_label in company_fields.items():
                        if label == field_label:
                            if key == "company_ssm_no":
                                details[key] = value if value != "-" else "Not Provided"
                            else:
                                details[key] = value
                            break
                    
                    if "website" in label and "social" not in label:
                        try:
                            link_elem = column.find_element(By.CSS_SELECTOR, "p a")
                            details["company_website"] = link_elem.get_attribute("href")
                        except NoSuchElementException:
                            details["company_website"] = value
                    
                    elif "social media" in label:
                        try:
                            social_links = column.find_elements(By.CSS_SELECTOR, "a[rel='nofollow']")
                            links = [link.get_attribute("href") for link in social_links if link.get_attribute("href")]
                            details["company_social_media"] = "; ".join(links) if links else "Not Found"
                        except NoSuchElementException:
                            continue
                
                except NoSuchElementException:
                    continue
            
            print(f"   ✓ Company info extracted")
        except Exception as e:
            print(f"   ⚠️ Company info error: {e}")
        
        # --- Extract Skills using OPTIMIZED METHOD (like foundit) ---
        try:
            print(f"🔍 Skills...")
            skills = []
            
            # Find skills section
            skills_section = soup.find('div', class_='space-y-6')
            if skills_section:
                # Check if this is the skills section by looking for "Skills" header
                header = skills_section.find('p', class_='font-bold')
                if header and 'skill' in header.get_text(strip=True).lower():
                    print(f"   ✓ Found Skills section")
                    
                    # Find the flex container with skills
                    flex_container = skills_section.find('div', class_='is-flex-wrap-wrap')
                    if flex_container:
                        # Direct search for skill spans - look for spans with border styling
                        skill_spans = flex_container.find_all('span', class_=lambda x: x and 'px-2' in str(x))
                        print(f"   Found {len(skill_spans)} potential skill elements")
                        
                        for span in skill_spans:
                            skill_text = span.get_text(strip=True)
                            # Filter: skills are typically short (< 50 chars) and not empty
                            if skill_text and len(skill_text) < 50 and skill_text not in skills:
                                skills.append(skill_text)
                                
            # Fallback: Search all flex-wrap-wrap containers
            if not skills:
                print(f"   Trying fallback method...")
                all_flex_containers = soup.find_all('div', class_='is-flex-wrap-wrap')
                
                for container in all_flex_containers:
                    skill_spans = container.find_all('span', class_=lambda x: x and 'px-2' in str(x))
                    
                    for span in skill_spans:
                        # Check if span has border styling (skills have this)
                        style = span.get('style', '')
                        classes = span.get('class', [])
                        
                        # Skills have: px-2, py-1, border-radius styling
                        if ('py-1' in classes or 'border-radius' in style):
                            skill_text = span.get_text(strip=True)
                            if skill_text and len(skill_text) < 50 and skill_text not in skills:
                                skills.append(skill_text)
                    
                    # If we found skills, stop searching
                    if skills:
                        break
            
            # Store skills (limit to first 15 like foundit does)
            details["skills"] = ", ".join(skills[:15]) if skills else "Not Found"
            
            if skills:
                print(f"   ✓ Found {len(skills)} skills: {', '.join(skills[:3])}...")
            else:
                print(f"   ⚠️ No skills found")
                    
        except Exception as e:
            print(f"   ⚠️ Skills error: {e}")
            details["skills"] = "Not Found"
        
        # --- Extract Working Location ---
        try:
            location_section = soup.find('div', {'id': 'job-locations'})
            if location_section:
                location_list = location_section.find('ul')
                if location_list:
                    locations = [li.get_text(strip=True) for li in location_list.find_all('li')]
                    details["working_location"] = "; ".join(locations)
        except Exception as e:
            pass
        
        # --- Extract Job Description sections ---
        try:
            job_desc_section = soup.find('div', {'id': 'job-description'})
            if job_desc_section:
                sections = job_desc_section.find_all('div', class_='space-y-4')
                
                for section in sections:
                    title_elem = section.find('p', class_='font-bold')
                    if title_elem:
                        title = title_elem.get_text(strip=True).lower()
                        content_div = section.find('div', class_='job-detail-text')
                        if content_div:
                            content_parts = []
                            for ul in content_div.find_all('ul'):
                                items = [li.get_text(strip=True) for li in ul.find_all('li')]
                                content_parts.extend(items)
                            for p in content_div.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            content = "\n".join(content_parts)
                            
                            if 'requirement' in title:
                                details["requirements"] = content
                            elif 'responsibilit' in title:
                                details["responsibilities"] = content
                            elif 'benefit' in title:
                                details["benefits"] = content
        except Exception as e:
            pass
        
        # --- Extract Dates ---
        try:
            date_text = soup.find('div', class_='text-base', string=lambda text: text and 'Posted' in text)
            if date_text:
                date_str = date_text.get_text(strip=True)
                if 'Posted' in date_str:
                    parts = date_str.split('•')
                    if len(parts) >= 1:
                        details["posted_date"] = parts[0].replace('Posted', '').strip()
                    if len(parts) >= 2:
                        details["closing_date"] = parts[1].replace('Closing', '').strip()
        except Exception as e:
            pass

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(1)
    
    return details

# ------------------------ MAIN SCRAPER ------------------------ #
max_pages = 2
all_jobs = []

for page in tqdm(range(1, max_pages + 1), desc="Pages", unit="page"):
    url = BASE_URL.format(page=page)
    print(f"\n📄 Page {page}")

    if not safe_get(url):
        print(f"❌ Failed to load page {page}")
        continue

    time.sleep(3)

    try:
        job_card_selector = "div.job-card-lite"
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector)))
        job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
        print(f"🔍 Found {len(job_cards)} jobs")

        for idx in tqdm(range(len(job_cards)), desc=f"Page {page}", unit="job", leave=False):
            try:
                job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
                job = job_cards[idx]
                
                job_title = job_url = company_name = location = salary = "Not Found"
                job_type = listing_date = category_text = job_id = "Not Found"
                subcategory_text = "N/A"
                
                try:
                    title_link = job.find_element(By.CSS_SELECTOR, "div.font-bold a.has-text-dark")
                    job_title = title_link.text.strip()
                    relative_url = title_link.get_attribute("href")
                    
                    if relative_url:
                        job_url = relative_url if relative_url.startswith("http") else "https://www.maukerja.my" + relative_url
                        if "/job/" in job_url:
                            job_id = job_url.split("/job/")[1].split("-")[0]
                except:
                    pass
                
                try:
                    company_elem = job.find_element(By.CSS_SELECTOR, "a[href*='/company/'] h2")
                    company_name = company_elem.text.strip()
                except:
                    pass
                
                try:
                    location_elems = job.find_elements(By.CSS_SELECTOR, "p.has-text-grey-dark a.joblocation-link")
                    location = ", ".join([e.text.strip() for e in location_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    salary_elem = job.find_element(By.CSS_SELECTOR, "span.has-text-primary.has-text-weight-semibold")
                    salary = salary_elem.text.strip()
                except:
                    pass
                
                try:
                    job_type_elems = job.find_elements(By.CSS_SELECTOR, "a.jobtype-link")
                    job_type = ", ".join([e.text.strip() for e in job_type_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    date_container = job.find_element(By.CSS_SELECTOR, "div.is-italic")
                    listing_date = date_container.text.strip().replace("Posted", "").strip()
                except:
                    pass
                
                try:
                    skill_elems = job.find_elements(By.CSS_SELECTOR, "div#jobCardSkills a")
                    skills = [e.text.strip() for e in skill_elems if e.text.strip()]
                    if skills:
                        category_text = skills[0]
                        subcategory_text = ", ".join(skills[1:]) if len(skills) > 1 else "N/A"
                except:
                    pass
                
                job_details = scrape_job_details(job_url, job_id) if job_url != "Not Found" else None
                
                job_description = "Not Found"
                if job_details:
                    parts = []
                    if job_details["requirements"] != "Not Found":
                        parts.append("REQUIREMENTS:\n" + job_details["requirements"])
                    if job_details["responsibilities"] != "Not Found":
                        parts.append("RESPONSIBILITIES:\n" + job_details["responsibilities"])
                    if parts:
                        job_description = "\n\n".join(parts)
                
                scraped_date = pd.to_datetime('today').strftime('%Y-%m-%d')
                
                all_jobs.append({
                    "Job ID": job_id,
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Company Type": job_details["company_type"] if job_details else "Not Found",
                    "Company Size": job_details["company_size"] if job_details else "Not Found",
                    "Industry": job_details["industry"] if job_details else "Not Found",
                    "Company SSM No": job_details["company_ssm_no"] if job_details else "Not Found",
                    "Job Vacancies": job_details["job_vacancies"] if job_details else "Not Found",
                    "Company Website": job_details["company_website"] if job_details else "Not Found",
                    "Company Social Media": job_details["company_social_media"] if job_details else "Not Found",
                    "Location": location,
                    "Working Location": job_details["working_location"] if job_details else "Not Found",
                    "Category": category_text,
                    "Subcategory": subcategory_text,
                    "Job Type": job_type,
                    "Salary": salary,
                    "Listing Date": listing_date,
                    "Posted Date": job_details["posted_date"] if job_details else "Not Found",
                    "Closing Date": job_details["closing_date"] if job_details else "Not Found",
                    "Job Description": job_description,
                    "Benefits": job_details["benefits"] if job_details else "Not Found",
                    "Skills": job_details["skills"] if job_details else "Not Found",
                    "Job URL": job_url,
                    "Scraped Date": scraped_date
                })
                
                print(f"✅ {job_title[:40]} @ {company_name[:30]}")
                
            except Exception as e:
                print(f"⚠️ Job {idx} error: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Page {page} error: {e}")
        continue

driver.quit()

# ------------------------ SAVE FILES ------------------------ #
df = pd.DataFrame(all_jobs)

timestamp = datetime.now().strftime('%d%B_%H%M%S')
excel_file = f"{timestamp}_maukerja_listings.xlsx"
csv_file = f"{timestamp}_maukerja_listings.csv"

try:
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")
except PermissionError:
    excel_file = f"{timestamp}_maukerja_v2.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")

try:
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"✅ CSV: {csv_file}")
except PermissionError:
    csv_file = f"{timestamp}_maukerja_v2.csv"
    try:
        df.to_csv(csv_file, index=False, encoding="utf-8-sig")
        print(f"✅ CSV: {csv_file}")
    except:
        print(f"⚠️ CSV failed")

print(f"\n🎉 Complete! Scraped {len(all_jobs)} jobs total")


🔐 MANUAL LOGIN REQUIRED
⏳ Maukerja homepage loaded.
✅ Cookies loaded from maukerja_cookies.json
🔄 Refreshing page with saved cookies...
✅ Page refreshed. Check if you're logged in.

📋 INSTRUCTIONS:
   1. Check if you're already logged in
   2. If NOT logged in: Click 'Sign In' and enter credentials
   3. Once logged in, return to this console

⚠️  DO NOT close the browser window!



✋ Press ENTER after you have successfully logged in...  


✅ Cookies saved to maukerja_cookies.json

✅ Login confirmed! Starting scraping process...



Pages:   0%|                                                                                   | 0/2 [00:00<?, ?page/s]


📄 Page 1
🔍 Found 31 jobs


...


Page 2:  97%|██████████████████████████████████████████████████████████████████████▋  | 30/31 [05:48<00:11, 11.67s/job]

✅ UI/UX @ Katch International Sdn Bhd
🔍 Company info...
   ✓ Company info extracted
🔍 Skills...
   Trying fallback method...
   ⚠️ No skills found



Pages: 100%|██████████████████████████████████████████████████████████████████████████| 2/2 [12:01<00:00, 360.81s/page]

✅ Marketing Executive @ Katch International Sdn Bhd



✅ Excel: 04December_103629_maukerja_listings.xlsx
✅ CSV: 04December_103629_maukerja_listings.csv

🎉 Complete! Scraped 62 jobs total


In [12]:
#v28

import time
import pandas as pd
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

# ------------------------ COOKIE MANAGEMENT ------------------------ #
COOKIE_FILE = "maukerja_cookies.json"

def save_cookies(driver, filepath=COOKIE_FILE):
    """Save cookies to a file after login."""
    with open(filepath, 'w') as f:
        json.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {filepath}")

def load_cookies(driver, filepath=COOKIE_FILE):
    """Load cookies from file to maintain login session."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            cookies = json.load(f)
            for cookie in cookies:
                if 'expiry' in cookie:
                    del cookie['expiry']
                try:
                    driver.add_cookie(cookie)
                except Exception as e:
                    print(f"⚠️ Could not add cookie: {e}")
        print(f"✅ Cookies loaded from {filepath}")
        return True
    else:
        print(f"⚠️ No cookie file found at {filepath}")
        return False

# ------------------------ SETUP ------------------------ #
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.page_load_strategy = "eager"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page={page}"

# ------------------------ MANUAL LOGIN ------------------------ #
print("\n" + "="*60)
print("🔐 MANUAL LOGIN REQUIRED")
print("="*60)

driver.get("https://www.maukerja.my/en")
print("⏳ Maukerja homepage loaded.")
time.sleep(3)

if load_cookies(driver):
    print("🔄 Refreshing page with saved cookies...")
    driver.refresh()
    time.sleep(5)
    print("✅ Page refreshed. Check if you're logged in.")
else:
    print("ℹ️  No saved cookies found. You need to log in manually.")

print("\n📋 INSTRUCTIONS:")
print("   1. Check if you're already logged in")
print("   2. If NOT logged in: Click 'Sign In' and enter credentials")
print("   3. Once logged in, return to this console")
print("\n⚠️  DO NOT close the browser window!")
print("="*60)

input("\n✋ Press ENTER after you have successfully logged in... ")

save_cookies(driver)
print("\n✅ Login confirmed! Starting scraping process...")
print("="*60 + "\n")
time.sleep(3)

# ------------------------ SAFE GET ------------------------ #
def safe_get(url, retries=3, wait_time=5):
    """Try loading a page with retries."""
    for i in range(retries):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"⚠️ Retry {i+1}/{retries} → {e}")
            time.sleep(wait_time)
    return False

# ------------------------ SCRAPE JOB DETAILS ------------------------ #
def scrape_job_details(job_url, job_id):
    """Scrape detailed job information."""
    details = {
        "working_location": "Not Found",
        "requirements": "Not Found",
        "responsibilities": "Not Found",
        "benefits": "Not Found",
        "skills": "Not Found",
        "posted_date": "Not Found",
        "closing_date": "Not Found",
        "company_type": "Not Found",
        "company_size": "Not Found",
        "industry": "Not Found",
        "company_ssm_no": "Not Found",
        "job_vacancies": "Not Found",
        "company_website": "Not Found",
        "company_social_media": "Not Found"
    }
    
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])

        if not safe_get(job_url):
            return details

        time.sleep(5)
        
        # Scroll to load all content
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # Get page source for BeautifulSoup
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # --- Extract Company Information ---
        try:
            print(f"🔍 Company info...")
            
            company_fields = {
                "company_type": "company type",
                "company_size": "company size",
                "industry": "industry",
                "company_ssm_no": "company ssm no",
                "job_vacancies": "job vacancies",
                "company_website": "company website"
            }
            
            company_columns = driver.find_elements(By.CSS_SELECTOR, ".has-background-grey90 .column")
            
            for column in company_columns:
                try:
                    label_elem = column.find_element(By.CSS_SELECTOR, "h3.has-text-weight-bold")
                    label = label_elem.text.strip().lower()
                    
                    value_elem = column.find_element(By.TAG_NAME, "p")
                    value = value_elem.text.strip()
                    
                    for key, field_label in company_fields.items():
                        if label == field_label:
                            if key == "company_ssm_no":
                                details[key] = value if value != "-" else "Not Provided"
                            else:
                                details[key] = value
                            break
                    
                    if "website" in label and "social" not in label:
                        try:
                            link_elem = column.find_element(By.CSS_SELECTOR, "p a")
                            details["company_website"] = link_elem.get_attribute("href")
                        except NoSuchElementException:
                            details["company_website"] = value
                    
                    elif "social media" in label:
                        try:
                            social_links = column.find_elements(By.CSS_SELECTOR, "a[rel='nofollow']")
                            links = [link.get_attribute("href") for link in social_links if link.get_attribute("href")]
                            details["company_social_media"] = "; ".join(links) if links else "Not Found"
                        except NoSuchElementException:
                            continue
                
                except NoSuchElementException:
                    continue
            
            print(f"   ✓ Company info extracted")
        except Exception as e:
            print(f"   ⚠️ Company info error: {e}")
        
        # --- Extract Skills - SIMPLE & DIRECT METHOD ---
        try:
            print(f"🔍 Skills...")
            skills = []
            
            # Step 1: Find the outer container - job-detail-text
            job_detail_containers = soup.find_all('div', class_='job-detail-text')
            print(f"   Found {len(job_detail_containers)} job-detail-text containers")
            
            # Step 2: Look inside each container for the flex-wrap div
            for container in job_detail_containers:
                flex_div = container.find('div', class_='is-flex-wrap-wrap')
                
                if flex_div:
                    print(f"   ✓ Found flex-wrap-wrap inside job-detail-text")
                    
                    # Step 3: Get ALL spans inside this flex div
                    all_spans = flex_div.find_all('span')
                    print(f"   Found {len(all_spans)} spans in flex-wrap")
                    
                    # Step 4: Extract text from each span
                    for span in all_spans:
                        skill_text = span.get_text(strip=True)
                        
                        # Only add if: has text, not too long, not duplicate
                        if skill_text and len(skill_text) > 0 and len(skill_text) < 100:
                            if skill_text not in skills:
                                skills.append(skill_text)
                                print(f"      • {skill_text}")
                    
                    # If we found skills in this container, stop searching
                    if skills:
                        break
            
            # Store results
            if skills:
                details["skills"] = ", ".join(skills)
                print(f"   ✅ Total: {len(skills)} skills extracted")
            else:
                details["skills"] = "Not Found"
                print(f"   ⚠️ No skills found")
                
        except Exception as e:
            print(f"   ❌ Error: {e}")
            details["skills"] = "Not Found"
        
        # --- Extract Working Location ---
        try:
            location_section = soup.find('div', {'id': 'job-locations'})
            if location_section:
                location_list = location_section.find('ul')
                if location_list:
                    locations = [li.get_text(strip=True) for li in location_list.find_all('li')]
                    details["working_location"] = "; ".join(locations)
        except Exception as e:
            pass
        
        # --- Extract Job Description sections ---
        try:
            job_desc_section = soup.find('div', {'id': 'job-description'})
            if job_desc_section:
                sections = job_desc_section.find_all('div', class_='space-y-4')
                
                for section in sections:
                    title_elem = section.find('p', class_='font-bold')
                    if title_elem:
                        title = title_elem.get_text(strip=True).lower()
                        content_div = section.find('div', class_='job-detail-text')
                        if content_div:
                            content_parts = []
                            for ul in content_div.find_all('ul'):
                                items = [li.get_text(strip=True) for li in ul.find_all('li')]
                                content_parts.extend(items)
                            for p in content_div.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            content = "\n".join(content_parts)
                            
                            if 'requirement' in title:
                                details["requirements"] = content
                            elif 'responsibilit' in title:
                                details["responsibilities"] = content
                            elif 'benefit' in title:
                                details["benefits"] = content
        except Exception as e:
            pass
        
        # --- Extract Dates ---
        try:
            date_text = soup.find('div', class_='text-base', string=lambda text: text and 'Posted' in text)
            if date_text:
                date_str = date_text.get_text(strip=True)
                if 'Posted' in date_str:
                    parts = date_str.split('•')
                    if len(parts) >= 1:
                        details["posted_date"] = parts[0].replace('Posted', '').strip()
                    if len(parts) >= 2:
                        details["closing_date"] = parts[1].replace('Closing', '').strip()
        except Exception as e:
            pass

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(1)
    
    return details

# ------------------------ MAIN SCRAPER ------------------------ #
max_pages = 2
all_jobs = []

for page in tqdm(range(1, max_pages + 1), desc="Pages", unit="page"):
    url = BASE_URL.format(page=page)
    print(f"\n📄 Page {page}")

    if not safe_get(url):
        print(f"❌ Failed to load page {page}")
        continue

    time.sleep(3)

    try:
        job_card_selector = "div.job-card-lite"
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector)))
        job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
        print(f"🔍 Found {len(job_cards)} jobs")

        for idx in tqdm(range(len(job_cards)), desc=f"Page {page}", unit="job", leave=False):
            try:
                job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
                job = job_cards[idx]
                
                job_title = job_url = company_name = location = salary = "Not Found"
                job_type = listing_date = category_text = job_id = "Not Found"
                subcategory_text = "N/A"
                
                try:
                    title_link = job.find_element(By.CSS_SELECTOR, "div.font-bold a.has-text-dark")
                    job_title = title_link.text.strip()
                    relative_url = title_link.get_attribute("href")
                    
                    if relative_url:
                        job_url = relative_url if relative_url.startswith("http") else "https://www.maukerja.my" + relative_url
                        if "/job/" in job_url:
                            job_id = job_url.split("/job/")[1].split("-")[0]
                except:
                    pass
                
                try:
                    company_elem = job.find_element(By.CSS_SELECTOR, "a[href*='/company/'] h2")
                    company_name = company_elem.text.strip()
                except:
                    pass
                
                try:
                    location_elems = job.find_elements(By.CSS_SELECTOR, "p.has-text-grey-dark a.joblocation-link")
                    location = ", ".join([e.text.strip() for e in location_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    salary_elem = job.find_element(By.CSS_SELECTOR, "span.has-text-primary.has-text-weight-semibold")
                    salary = salary_elem.text.strip()
                except:
                    pass
                
                try:
                    job_type_elems = job.find_elements(By.CSS_SELECTOR, "a.jobtype-link")
                    job_type = ", ".join([e.text.strip() for e in job_type_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    date_container = job.find_element(By.CSS_SELECTOR, "div.is-italic")
                    listing_date = date_container.text.strip().replace("Posted", "").strip()
                except:
                    pass
                
                try:
                    skill_elems = job.find_elements(By.CSS_SELECTOR, "div#jobCardSkills a")
                    skills = [e.text.strip() for e in skill_elems if e.text.strip()]
                    if skills:
                        category_text = skills[0]
                        subcategory_text = ", ".join(skills[1:]) if len(skills) > 1 else "N/A"
                except:
                    pass
                
                job_details = scrape_job_details(job_url, job_id) if job_url != "Not Found" else None
                
                job_description = "Not Found"
                if job_details:
                    parts = []
                    if job_details["requirements"] != "Not Found":
                        parts.append("REQUIREMENTS:\n" + job_details["requirements"])
                    if job_details["responsibilities"] != "Not Found":
                        parts.append("RESPONSIBILITIES:\n" + job_details["responsibilities"])
                    if parts:
                        job_description = "\n\n".join(parts)
                
                scraped_date = pd.to_datetime('today').strftime('%Y-%m-%d')
                
                all_jobs.append({
                    "Job ID": job_id,
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Company Type": job_details["company_type"] if job_details else "Not Found",
                    "Company Size": job_details["company_size"] if job_details else "Not Found",
                    "Industry": job_details["industry"] if job_details else "Not Found",
                    "Company SSM No": job_details["company_ssm_no"] if job_details else "Not Found",
                    "Job Vacancies": job_details["job_vacancies"] if job_details else "Not Found",
                    "Company Website": job_details["company_website"] if job_details else "Not Found",
                    "Company Social Media": job_details["company_social_media"] if job_details else "Not Found",
                    "Location": location,
                    "Working Location": job_details["working_location"] if job_details else "Not Found",
                    "Category": category_text,
                    "Subcategory": subcategory_text,
                    "Job Type": job_type,
                    "Salary": salary,
                    "Listing Date": listing_date,
                    "Posted Date": job_details["posted_date"] if job_details else "Not Found",
                    "Closing Date": job_details["closing_date"] if job_details else "Not Found",
                    "Job Description": job_description,
                    "Benefits": job_details["benefits"] if job_details else "Not Found",
                    "Skills": job_details["skills"] if job_details else "Not Found",
                    "Job URL": job_url,
                    "Scraped Date": scraped_date
                })
                
                print(f"✅ {job_title[:40]} @ {company_name[:30]}")
                
            except Exception as e:
                print(f"⚠️ Job {idx} error: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Page {page} error: {e}")
        continue

driver.quit()

# ------------------------ SAVE FILES ------------------------ #
df = pd.DataFrame(all_jobs)

timestamp = datetime.now().strftime('%d%B_%H%M%S')
excel_file = f"{timestamp}_maukerja_listings.xlsx"
csv_file = f"{timestamp}_maukerja_listings.csv"

try:
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")
except PermissionError:
    excel_file = f"{timestamp}_maukerja_v2.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")

try:
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"✅ CSV: {csv_file}")
except PermissionError:
    csv_file = f"{timestamp}_maukerja_v2.csv"
    try:
        df.to_csv(csv_file, index=False, encoding="utf-8-sig")
        print(f"✅ CSV: {csv_file}")
    except:
        print(f"⚠️ CSV failed")

print(f"\n🎉 Complete! Scraped {len(all_jobs)} jobs total")


🔐 MANUAL LOGIN REQUIRED
⏳ Maukerja homepage loaded.
✅ Cookies loaded from maukerja_cookies.json
🔄 Refreshing page with saved cookies...
✅ Page refreshed. Check if you're logged in.

📋 INSTRUCTIONS:
   1. Check if you're already logged in
   2. If NOT logged in: Click 'Sign In' and enter credentials
   3. Once logged in, return to this console

⚠️  DO NOT close the browser window!



✋ Press ENTER after you have successfully logged in...  


✅ Cookies saved to maukerja_cookies.json

✅ Login confirmed! Starting scraping process...



Pages:   0%|                                                                                   | 0/2 [00:00<?, ?page/s]


📄 Page 1
🔍 Found 31 jobs


...


Page 2:  97%|██████████████████████████████████████████████████████████████████████▋  | 30/31 [05:56<00:11, 11.99s/job]

✅ Retail Associate (Mango, Ioi City Mall) @ Mango
🔍 Company info...
   ✓ Company info extracted
🔍 Skills...
   Found 3 job-detail-text containers
   ⚠️ No skills found



Pages: 100%|██████████████████████████████████████████████████████████████████████████| 2/2 [12:11<00:00, 365.89s/page]

✅ Business Development Executive @ Otter Barista



✅ Excel: 04December_105223_maukerja_listings.xlsx
✅ CSV: 04December_105223_maukerja_listings.csv

🎉 Complete! Scraped 62 jobs total


In [1]:
#V34

import time
import pandas as pd
import json
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime

from datetime import datetime, timedelta
import re

# ------------------------ COOKIE MANAGEMENT ------------------------ #
COOKIE_FILE = "maukerja_cookies.json"

def save_cookies(driver, filepath=COOKIE_FILE):
    """Save cookies to a file after login."""
    with open(filepath, 'w') as f:
        json.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {filepath}")

def load_cookies(driver, filepath=COOKIE_FILE):
    """Load cookies from file to maintain login session."""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            cookies = json.load(f)
            for cookie in cookies:
                if 'expiry' in cookie:
                    del cookie['expiry']
                try:
                    driver.add_cookie(cookie)
                except Exception as e:
                    print(f"⚠️ Could not add cookie: {e}")
        print(f"✅ Cookies loaded from {filepath}")
        return True
    else:
        print(f"⚠️ No cookie file found at {filepath}")
        return False

# ------------------------ DATE CONVERSION ------------------------ #
def convert_posted_date_to_format(posted_text):
    """Convert 'Posted X ago' to DD/MM/YYYY format"""
    if not posted_text or posted_text == "Not Found":
        return "Not Found"
    
    try:
        posted_text = posted_text.lower().strip()
        today = datetime.now()
        
        # Handle "today"
        if 'today' in posted_text:
            return today.strftime('%d/%m/%Y')
        
        # Handle "yesterday"
        if 'yesterday' in posted_text:
            return (today - timedelta(days=1)).strftime('%d/%m/%Y')
        
        # Extract numbers from text
        numbers = re.findall(r'\d+', posted_text)
        if not numbers:
            return posted_text
        
        value = int(numbers[0])
        
        # Calculate date based on time unit
        if 'minute' in posted_text or 'hour' in posted_text:
            return today.strftime('%d/%m/%Y')
        elif 'day' in posted_text:
            return (today - timedelta(days=value)).strftime('%d/%m/%Y')
        elif 'week' in posted_text:
            return (today - timedelta(weeks=value)).strftime('%d/%m/%Y')
        elif 'month' in posted_text:
            return (today - timedelta(days=value*30)).strftime('%d/%m/%Y')
        else:
            return posted_text
            
    except Exception as e:
        return posted_text

# ------------------------ SETUP ------------------------ #
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.page_load_strategy = "eager"

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 20)

BASE_URL = "https://www.maukerja.my/en/jobsearch/jobs-in-malaysia?sortBy=date&created=now-30d&page={page}"

# ------------------------ MANUAL LOGIN ------------------------ #
print("\n" + "="*60)
print("🔐 MANUAL LOGIN REQUIRED")
print("="*60)

driver.get("https://www.maukerja.my/en")
print("⏳ Maukerja homepage loaded.")
time.sleep(3)

if load_cookies(driver):
    print("🔄 Refreshing page with saved cookies...")
    driver.refresh()
    time.sleep(5)
    print("✅ Page refreshed. Check if you're logged in.")
else:
    print("ℹ️  No saved cookies found. You need to log in manually.")

print("\n📋 INSTRUCTIONS:")
print("   1. Check if you're already logged in")
print("   2. If NOT logged in: Click 'Sign In' and enter credentials")
print("   3. Once logged in, return to this console")
print("\n⚠️  DO NOT close the browser window!")
print("="*60)

input("\n✋ Press ENTER after you have successfully logged in... ")

save_cookies(driver)
print("\n✅ Login confirmed! Starting scraping process...")
print("="*60 + "\n")
time.sleep(3)

# ------------------------ SAFE GET ------------------------ #
def safe_get(url, retries=3, wait_time=5):
    """Try loading a page with retries."""
    for i in range(retries):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"⚠️ Retry {i+1}/{retries} → {e}")
            time.sleep(wait_time)
    return False

# ------------------------ SCRAPE JOB DETAILS ------------------------ #
def scrape_job_details(job_url, job_id):
    """Scrape detailed job information."""
    details = {
        "working_location": "Not Found",
        "requirements": "Not Found",
        "responsibilities": "Not Found",
        "benefits": "Not Found",
        "skills": "Not Found",
        "posted_date": "Not Found",
        "closing_date": "Not Found",
        "company_type": "Not Found",
        "company_size": "Not Found",
        "industry": "Not Found",
        "company_ssm_no": "Not Found",
        "job_vacancies": "Not Found",
        "company_website": "Not Found",
        "company_social_media": "Not Found"
    }
    
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])

        if not safe_get(job_url):
            return details

        time.sleep(3)  # Reduced from 5 to 3 seconds
        
        # Faster scroll - still human-like
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)  # Reduced from 2 to 1 second
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(0.5)  # Reduced from 1 to 0.5 seconds
        
        # Get page source for BeautifulSoup
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # --- Extract Company Information ---
        try:
            print(f"🔍 Company info...")
            
            company_fields = {
                "company_type": "company type",
                "company_size": "company size",
                "industry": "industry",
                "company_ssm_no": "company ssm no",
                "job_vacancies": "job vacancies",
                "company_website": "company website"
            }
            
            company_columns = driver.find_elements(By.CSS_SELECTOR, ".has-background-grey90 .column")
            
            for column in company_columns:
                try:
                    label_elem = column.find_element(By.CSS_SELECTOR, "h3.has-text-weight-bold")
                    label = label_elem.text.strip().lower()
                    
                    value_elem = column.find_element(By.TAG_NAME, "p")
                    value = value_elem.text.strip()
                    
                    for key, field_label in company_fields.items():
                        if label == field_label:
                            if key == "company_ssm_no":
                                details[key] = value if value != "-" else "Not Provided"
                            else:
                                details[key] = value
                            break
                    
                    if "website" in label and "social" not in label:
                        try:
                            link_elem = column.find_element(By.CSS_SELECTOR, "p a")
                            details["company_website"] = link_elem.get_attribute("href")
                        except NoSuchElementException:
                            details["company_website"] = value
                    
                    elif "social media" in label:
                        try:
                            social_links = column.find_elements(By.CSS_SELECTOR, "a[rel='nofollow']")
                            links = [link.get_attribute("href") for link in social_links if link.get_attribute("href")]
                            details["company_social_media"] = "; ".join(links) if links else "Not Found"
                        except NoSuchElementException:
                            continue
                
                except NoSuchElementException:
                    continue
            
            print(f"   ✓ Company info extracted")
        except Exception as e:
            print(f"   ⚠️ Company info error: {e}")
        
        # --- Extract Skills - SIMPLE & DIRECT METHOD ---
        try:
            print(f"🔍 Skills...")
            skills = []
            
            # Step 1: Find the outer container - job-detail-text
            job_detail_containers = soup.find_all('div', class_='job-detail-text')
            print(f"   Found {len(job_detail_containers)} job-detail-text containers")
            
            # Step 2: Look inside each container for the flex-wrap div
            for container in job_detail_containers:
                flex_div = container.find('div', class_='is-flex-wrap-wrap')
                
                if flex_div:
                    print(f"   ✓ Found flex-wrap-wrap inside job-detail-text")
                    
                    # Step 3: Get ALL spans inside this flex div
                    all_spans = flex_div.find_all('span')
                    print(f"   Found {len(all_spans)} spans in flex-wrap")
                    
                    # Step 4: Extract text from each span
                    for span in all_spans:
                        skill_text = span.get_text(strip=True)
                        
                        # Only add if: has text, not too long, not duplicate
                        if skill_text and len(skill_text) > 0 and len(skill_text) < 100:
                            if skill_text not in skills:
                                skills.append(skill_text)
                                print(f"      • {skill_text}")
                    
                    # If we found skills in this container, stop searching
                    if skills:
                        break
            
            # Store results
            if skills:
                details["skills"] = ", ".join(skills)
                print(f"   ✅ Total: {len(skills)} skills extracted")
            else:
                details["skills"] = "Not Found"
                print(f"   ⚠️ No skills found")
                
        except Exception as e:
            print(f"   ❌ Error: {e}")
            details["skills"] = "Not Found"
        
        # --- Extract Working Location ---
        try:
            location_section = soup.find('div', {'id': 'job-locations'})
            if location_section:
                location_list = location_section.find('ul')
                if location_list:
                    locations = [li.get_text(strip=True) for li in location_list.find_all('li')]
                    details["working_location"] = "; ".join(locations)
        except Exception as e:
            pass
        
        # --- Extract Job Description sections ---
        try:
            job_desc_section = soup.find('div', {'id': 'job-description'})
            if job_desc_section:
                sections = job_desc_section.find_all('div', class_='space-y-4')
                
                for section in sections:
                    title_elem = section.find('p', class_='font-bold')
                    if title_elem:
                        title = title_elem.get_text(strip=True).lower()
                        content_div = section.find('div', class_='job-detail-text')
                        if content_div:
                            content_parts = []
                            for ul in content_div.find_all('ul'):
                                items = [li.get_text(strip=True) for li in ul.find_all('li')]
                                content_parts.extend(items)
                            for p in content_div.find_all('p'):
                                text = p.get_text(strip=True)
                                if text:
                                    content_parts.append(text)
                            content = "\n".join(content_parts)
                            
                            if 'requirement' in title:
                                details["requirements"] = content
                            elif 'responsibilit' in title:
                                details["responsibilities"] = content
                            elif 'benefit' in title:
                                details["benefits"] = content
        except Exception as e:
            pass
        
        # --- Extract Dates ---
        try:
            date_text = soup.find('div', class_='text-base', string=lambda text: text and 'Posted' in text)
            if date_text:
                date_str = date_text.get_text(strip=True)
                if 'Posted' in date_str:
                    parts = date_str.split('•')
                    if len(parts) >= 1:
                        details["posted_date"] = parts[0].replace('Posted', '').strip()
                    if len(parts) >= 2:
                        details["closing_date"] = parts[1].replace('Closing', '').strip()
        except Exception as e:
            pass

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        time.sleep(0.5)  # Reduced from 1 to 0.5 seconds
    
    return details

# ------------------------ MAIN SCRAPER ------------------------ #
max_pages = 70  # Changed from 2 to 70 pages
all_jobs = []

for page in tqdm(range(1, max_pages + 1), desc="Pages", unit="page"):
    url = BASE_URL.format(page=page)
    print(f"\n📄 Page {page}")

    if not safe_get(url):
        print(f"❌ Failed to load page {page}")
        continue

    time.sleep(2)  # Reduced from 3 to 2 seconds

    try:
        job_card_selector = "div.job-card-lite"
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector)))
        job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
        print(f"🔍 Found {len(job_cards)} jobs")

        for idx in tqdm(range(len(job_cards)), desc=f"Page {page}", unit="job", leave=False):
            try:
                job_cards = driver.find_elements(By.CSS_SELECTOR, job_card_selector)
                job = job_cards[idx]
                
                job_title = job_url = company_name = location = salary = "Not Found"
                job_type = listing_date = category_text = job_id = "Not Found"
                subcategory_text = "N/A"
                
                try:
                    title_link = job.find_element(By.CSS_SELECTOR, "div.font-bold a.has-text-dark")
                    job_title = title_link.text.strip()
                    relative_url = title_link.get_attribute("href")
                    
                    if relative_url:
                        job_url = relative_url if relative_url.startswith("http") else "https://www.maukerja.my" + relative_url
                        if "/job/" in job_url:
                            job_id = job_url.split("/job/")[1].split("-")[0]
                except:
                    pass
                
                try:
                    company_elem = job.find_element(By.CSS_SELECTOR, "a[href*='/company/'] h2")
                    company_name = company_elem.text.strip()
                except:
                    pass
                
                try:
                    location_elems = job.find_elements(By.CSS_SELECTOR, "p.has-text-grey-dark a.joblocation-link")
                    location = ", ".join([e.text.strip() for e in location_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    salary_elem = job.find_element(By.CSS_SELECTOR, "span.has-text-primary.has-text-weight-semibold")
                    salary = salary_elem.text.strip()
                except:
                    pass
                
                try:
                    job_type_elems = job.find_elements(By.CSS_SELECTOR, "a.jobtype-link")
                    job_type = ", ".join([e.text.strip() for e in job_type_elems if e.text.strip()])
                except:
                    pass
                
                try:
                    date_container = job.find_element(By.CSS_SELECTOR, "div.is-italic")
                    listing_date = date_container.text.strip().replace("Posted", "").strip()
                except:
                    pass
                
                try:
                    skill_elems = job.find_elements(By.CSS_SELECTOR, "div#jobCardSkills a")
                    skills = [e.text.strip() for e in skill_elems if e.text.strip()]
                    if skills:
                        category_text = skills[0]
                        subcategory_text = ", ".join(skills[1:]) if len(skills) > 1 else "N/A"
                except:
                    pass
                
                job_details = scrape_job_details(job_url, job_id) if job_url != "Not Found" else None
                
                job_description = "Not Found"
                if job_details:
                    parts = []
                    if job_details["requirements"] != "Not Found":
                        parts.append("REQUIREMENTS:\n" + job_details["requirements"])
                    if job_details["responsibilities"] != "Not Found":
                        parts.append("RESPONSIBILITIES:\n" + job_details["responsibilities"])
                    if parts:
                        job_description = "\n\n".join(parts)
                
                # Convert dates to DD/MM/YYYY format
                scraped_date = datetime.now().strftime('%d/%m/%Y')
                posted_date_formatted = convert_posted_date_to_format(job_details["posted_date"]) if job_details else "Not Found"
                
                all_jobs.append({
                    "Job ID": job_id,
                    "Job Title": job_title,
                    "Company Name": company_name,
                    "Company Type": job_details["company_type"] if job_details else "Not Found",
                    "Company Size": job_details["company_size"] if job_details else "Not Found",
                    "Industry": job_details["industry"] if job_details else "Not Found",
                    "Company SSM No": job_details["company_ssm_no"] if job_details else "Not Found",
                    "Job Vacancies": job_details["job_vacancies"] if job_details else "Not Found",
                    "Company Website": job_details["company_website"] if job_details else "Not Found",
                    "Company Social Media": job_details["company_social_media"] if job_details else "Not Found",
                    "Location": location,
                    "Working Location": job_details["working_location"] if job_details else "Not Found",
                    "Category": category_text,
                    "Subcategory": subcategory_text,
                    "Job Type": job_type,
                    "Salary": salary,
                    "Listing Date": listing_date,
                    "Posted Date": posted_date_formatted,
                    "Closing Date": job_details["closing_date"] if job_details else "Not Found",
                    "Job Description": job_description,
                    "Benefits": job_details["benefits"] if job_details else "Not Found",
                    "Skills": job_details["skills"] if job_details else "Not Found",
                    "Job URL": job_url,
                    "Scraped Date": scraped_date
                })
                
                print(f"✅ {job_title[:40]} @ {company_name[:30]}")
                
            except Exception as e:
                print(f"⚠️ Job {idx} error: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Page {page} error: {e}")
        continue

driver.quit()

# ------------------------ SAVE FILES ------------------------ #
df = pd.DataFrame(all_jobs)

timestamp = datetime.now().strftime('%d%B_%H%M%S')
excel_file = f"{timestamp}_maukerja_listings.xlsx"
csv_file = f"{timestamp}_maukerja_listings.csv"

try:
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")
except PermissionError:
    excel_file = f"{timestamp}_maukerja_v2.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"\n✅ Excel: {excel_file}")

try:
    df.to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"✅ CSV: {csv_file}")
except PermissionError:
    csv_file = f"{timestamp}_maukerja_v2.csv"
    try:
        df.to_csv(csv_file, index=False, encoding="utf-8-sig")
        print(f"✅ CSV: {csv_file}")
    except:
        print(f"⚠️ CSV failed")

print(f"\n🎉 Complete! Scraped {len(all_jobs)} jobs total")


🔐 MANUAL LOGIN REQUIRED
⏳ Maukerja homepage loaded.
✅ Cookies loaded from maukerja_cookies.json
🔄 Refreshing page with saved cookies...
✅ Page refreshed. Check if you're logged in.

📋 INSTRUCTIONS:
   1. Check if you're already logged in
   2. If NOT logged in: Click 'Sign In' and enter credentials
   3. Once logged in, return to this console

⚠️  DO NOT close the browser window!



✋ Press ENTER after you have successfully logged in...  


✅ Cookies saved to maukerja_cookies.json

✅ Login confirmed! Starting scraping process...



Pages:   0%|                                                                                  | 0/70 [00:00<?, ?page/s]


📄 Page 1
🔍 Found 31 jobs


...


Page 70:  97%|█████████████████████████████████████████████████████████████████████▋  | 30/31 [03:47<00:07,  7.76s/job]

✅ Environmental Officer @ Widad Builders Sdn Bhd
🔍 Company info...
   ✓ Company info extracted
🔍 Skills...
   Found 3 job-detail-text containers
   ⚠️ No skills found



Pages: 100%|██████████████████████████████████████████████████████████████████████| 70/70 [4:47:28<00:00, 246.40s/page]

✅ HR Assistant (Mandarin) @ Netforge Solution Sdn. Bhd.


IllegalCharacterError: REQUIREMENTS:
Strong communication and organizational skills.
Attention to detail and ability to multitask.
Positive attitude and willingness to learn.
Proficient in MS Office (Word, Excel, PowerPoint)
Experience with accounting software (SQL, QnE etc) is an advantage.
Fresh Graduatedare encourage to apply

RESPONSIBILITIES:
Attend to telephone, email and WhatsApp enquiries and walk-incustomers.
Communicate with sales personnel on order and enquiries.
Coordinating with warehouse team on packing and delivery.
Assist in Invoicing, quotations and filing of document.
Assist in Purchasing, Shipping and Stock Management.
Carry out general accounting and administrative tasks.
Assist in updating and maintaining website and online marketing and preparation of catalogue.
Any other duties as may be assigned by the management from time to time. cannot be used in worksheets.